# Binary-Only Evidence Consolidation

This notebook documents the first **unified evidence-consolidation** experiments for dyadic conversational anomaly detection.

The preceding isolated branches showed that useful information existed separately for:

- **semantic compatibility** — especially NORMAL vs Wrong Partner,
- **temporal coordination** — especially NORMAL vs Lag Partner,
- **spoken participation** — especially Silent Partner.

The next question was whether these heterogeneous evidence sources could be combined into a **single conversation-level binary decision**:

- `NORMAL`
- `ANOMALOUS`

No anomaly subtype is predicted.

## Common consolidation evidence

Both experiments retained in this notebook use the same 400-case development set:

- 100 NORMAL,
- 100 LAG,
- 100 WRONG PARTNER,
- 100 SILENT PARTNER.

The reasoner receives the same underlying evidence in both experiments:

### Participation evidence
- participant-level `speaks`,
- backchannel-filtered participant turn sequences.

### Local temporal evidence
- signed A-end-to-B-start response offsets and their statistics,
- filtered overlap,
- the participant turn structure.

### Global temporal evidence
- estimated Participant-B correction shift / lateness,
- alignment-score gain,
- bilateral alignment-event support,
- event coverage.

### Semantic evidence
- coarse summaries for two aligned 60-second segments,
- focused summaries for the same segments.

### Frozen temporal reference
- statistics calculated only from separate NORMAL reference conversations.

The model does **not** receive case identifiers, pairing labels, anomaly type, gold labels, file paths, or other hidden case metadata.

## Experimental progression

This notebook intentionally retains two binary-only prompt designs.

### Experiment 1 — Initial binary consolidation prompt

The first prompt described the general characteristics of a coherent dyadic interaction, but it did not force the model to assess participation, semantics, and temporal coordination as independent requirements.

This produced a strong tendency to preserve cases as NORMAL whenever some evidence appeared conversationally plausible. In particular, semantic compatibility could compensate for or overshadow temporal deviations.

The resulting performance was:

| Family | Correct |
|---|---:|
| NORMAL | 99/100 |
| LAG | 7/100 |
| WRONG PARTNER | 45/100 |
| SILENT PARTNER | 100/100 |
| **Overall** | **62.75%** |

The failure is diagnostically useful: simply providing all evidence sources does not guarantee that the model will use each of them appropriately.

### Experiment 2 — Binary-only consolidation with independent normality requirements

The second experiment changes **only the prompt-level decision policy**.

A NORMAL interaction is now required to satisfy three independent conditions:

1. valid spoken participation,
2. semantic conversational compatibility,
3. temporal coordination.

A strong, reliable failure in any one dimension is sufficient for an `ANOMALOUS` prediction. In particular, semantic compatibility is explicitly prevented from cancelling reliable temporal evidence.

All data, features, frozen references, model settings, decoding settings, and the binary-only output schema remain unchanged.

This revised binary-only configuration produces the result reported in the thesis:

| Format | NORMAL | LAG | WRONG | SILENT | Accuracy |
|---|---:|---:|---:|---:|---:|
| **Binary only** | **72/100** | **54/100** | **95/100** | **100/100** | **80.25%** |

This notebook therefore records the transition from an under-specified unified prompt to the **Binary-only consolidation baseline** used in the later comparison of output formats.

> **Reproducibility note:** all retained code cells and saved outputs are preserved exactly as executed. Only Markdown presentation has been rewritten. All later binary prompt variants from the original development notebook have been removed from this public version.

## 1. Install Dependencies

Install the Qwen2.5-Omni and evaluation dependencies used for text-only evidence consolidation.

In [ ]:
!pip uninstall -y transformers
!pip install -U "transformers>=4.57.0" accelerate bitsandbytes sentencepiece
!pip install -U qwen-omni-utils decord ffmpeg-python
!apt-get update -qq
!apt-get install -y -qq ffmpeg

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 123.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 131.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 75.2 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2. Mount Google Drive and Configure Consolidation Artifacts

The consolidation stage operates on precomputed participant-centric evidence rather than raw video.

This cell locates:

- the frozen temporal reference statistics,
- the final 400-case structured consolidation database,
- the Qwen2.5-Omni checkpoint,
- the persistent experiment output directories.

The 400-case database contains the semantic, temporal, and participation evidence generated upstream by the isolated branches and common preprocessing pipeline.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


from pathlib import Path
from collections import Counter
import copy
import json

import pandas as pd
from IPython.display import display


OUT_DIR = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec"
)


REFERENCE_BASE_STATS_PATH = (
    OUT_DIR
    / "frozen_reference_base_statistics.json"
)


REFERENCE_SHIFT_STATS_PATH = (
    OUT_DIR
    / "frozen_reference_global_shift_statistics.json"
)


FINAL_DATABASE_PATH = (
    OUT_DIR
    / (
        "consolidation_all_400_cases_with_"
        "temporal_and_semantic_summaries.json"
    )
)


MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

MODEL_PATH = Path(
    "/content/drive/MyDrive/Qwen2.5-Omni-7B"
)


required_paths = {
    "Frozen base statistics": (
        REFERENCE_BASE_STATS_PATH
    ),

    "Frozen global-shift statistics": (
        REFERENCE_SHIFT_STATS_PATH
    ),

    "Final 400-case database": (
        FINAL_DATABASE_PATH
    ),

    "Local Qwen checkpoint": (
        MODEL_PATH
    ),
}


print("=" * 88)
print("ARTIFACT PATH AUDIT")
print("=" * 88)

for name, path in required_paths.items():

    print(
        f"{name}:",
        path,
    )

    print(
        "  exists:",
        path.exists(),
    )


assert OUT_DIR.exists(), (
    f"Project directory not found: {OUT_DIR}"
)

assert REFERENCE_BASE_STATS_PATH.exists(), (
    "Frozen base-statistics file not found: "
    f"{REFERENCE_BASE_STATS_PATH}"
)

assert REFERENCE_SHIFT_STATS_PATH.exists(), (
    "Frozen global-shift-statistics file not found: "
    f"{REFERENCE_SHIFT_STATS_PATH}"
)

assert FINAL_DATABASE_PATH.exists(), (
    "Final consolidation database not found: "
    f"{FINAL_DATABASE_PATH}"
)

assert MODEL_PATH.exists(), (
    "Local Qwen checkpoint not found: "
    f"{MODEL_PATH}"
)

Mounted at /content/drive
ARTIFACT PATH AUDIT
Frozen base statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_base_statistics.json
  exists: True
Frozen global-shift statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_global_shift_statistics.json
  exists: True
Final 400-case database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
  exists: True
Local Qwen checkpoint: /content/drive/MyDrive/Qwen2.5-Omni-7B
  exists: True


## 3. Load the Frozen NORMAL Temporal Reference

The stored temporal-reference files contain several profiles, but the initial binary consolidation setup intentionally exposes only the **NORMAL** reference profile.

These statistics provide a soft description of ordinary temporal coordination. They are not hard thresholds and they are never computed from the evaluated cases themselves.

In [ ]:
# ============================================================
# LOAD FROZEN STATISTICS AND RETAIN NORMAL ONLY
# ============================================================

def load_json_list(
    path,
):
    data = json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


    assert isinstance(
        data,
        list,
    ), (
        f"Expected a JSON list in {path}"
    )


    return data


frozen_base_records_all_profiles = (
    load_json_list(
        REFERENCE_BASE_STATS_PATH
    )
)


frozen_shift_records_all_profiles = (
    load_json_list(
        REFERENCE_SHIFT_STATS_PATH
    )
)


base_profiles = [
    str(
        record.get(
            "reference_profile"
        )
    )

    for record
    in frozen_base_records_all_profiles
]


shift_profiles = [
    str(
        record.get(
            "reference_profile"
        )
    )

    for record
    in frozen_shift_records_all_profiles
]


assert len(
    base_profiles
) == len(
    set(
        base_profiles
    )
)


assert len(
    shift_profiles
) == len(
    set(
        shift_profiles
    )
)


assert "NORMAL" in base_profiles
assert "NORMAL" in shift_profiles


normal_base_matches = [
    record

    for record
    in frozen_base_records_all_profiles

    if str(
        record.get(
            "reference_profile"
        )
    ) == "NORMAL"
]


normal_shift_matches = [
    record

    for record
    in frozen_shift_records_all_profiles

    if str(
        record.get(
            "reference_profile"
        )
    ) == "NORMAL"
]


assert len(
    normal_base_matches
) == 1


assert len(
    normal_shift_matches
) == 1


frozen_normal_base_statistics = copy.deepcopy(
    normal_base_matches[0]
)


frozen_normal_shift_statistics = copy.deepcopy(
    normal_shift_matches[0]
)


assert (
    frozen_normal_base_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


assert (
    frozen_normal_shift_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


# This is the only frozen reference object that should later
# be passed to the binary-only prompt constructor.
FROZEN_NORMAL_REFERENCE = {
    "base_statistics": (
        frozen_normal_base_statistics
    ),

    "global_shift_statistics": (
        frozen_normal_shift_statistics
    ),
}


normal_base_display_df = pd.DataFrame([
    {
        "metric": key,
        "value": value,
    }

    for key, value
    in frozen_normal_base_statistics.items()

    if key != "reference_profile"
])


normal_shift_display_df = pd.DataFrame([
    {
        "metric": key,
        "value": value,
    }

    for key, value
    in frozen_normal_shift_statistics.items()

    if key != "reference_profile"
])


print("=" * 88)
print("FROZEN REFERENCE FILES LOADED")
print("=" * 88)

print(
    "Profiles stored in base-statistics file:",
    base_profiles,
)

print(
    "Profiles stored in global-shift file:",
    shift_profiles,
)


print("\n" + "=" * 88)
print("FROZEN NORMAL BASE TEMPORAL STATISTICS")
print("=" * 88)

display(
    normal_base_display_df
)


print("\n" + "=" * 88)
print("FROZEN NORMAL GLOBAL-SHIFT STATISTICS")
print("=" * 88)

display(
    normal_shift_display_df
)


print(
    "\nOnly NORMAL frozen references retained:",
    list(
        FROZEN_NORMAL_REFERENCE.keys()
    ),
)

FROZEN REFERENCE FILES LOADED
Profiles stored in base-statistics file: ['NORMAL', 'LAG_1', 'LAG_2', 'LAG_3']
Profiles stored in global-shift file: ['NORMAL', 'LAG_1', 'LAG_2', 'LAG_3']

FROZEN NORMAL BASE TEMPORAL STATISTICS


,metric,value
0,num_original_A_turns,11.800000
1,num_original_B_turns,11.080000
2,num_removed_A_backchannels,1.740000
3,num_removed_B_backchannels,1.660000
4,num_filtered_A_turns,10.060000
5,num_filtered_B_turns,9.420000
6,num_offsets,5.760000
7,num_negative,2.040000
8,num_positive,3.720000
9,offset_mean,0.302128



FROZEN NORMAL GLOBAL-SHIFT STATISTICS


,metric,value
0,best_B_correction_shift_seconds__mean,-0.230000
1,best_B_correction_shift_seconds__median,-0.000000
2,estimated_B_lateness_seconds__mean,0.500000
3,estimated_B_lateness_seconds__median,0.000000
4,alignment_score_gain_vs_zero__mean,0.023376
5,alignment_score_gain_vs_zero__median,0.009105
6,best_num_bilateral_events__mean,11.940000
7,best_event_coverage__mean,0.595175



Only NORMAL frozen references retained: ['base_statistics', 'global_shift_statistics']


## 4. Load and Audit the 400-Case Consolidation Database

The structured database contains exactly:

- 100 NORMAL cases,
- 100 LAG cases,
- 100 WRONG PARTNER cases,
- 100 SILENT PARTNER cases.

The audit verifies the expected binary labels and checks the consistency of participant-level participation, temporal, and semantic fields before any model inference is performed.

The same 400 cases are used for both binary-only prompt experiments below.

In [ ]:
# ============================================================
# LOAD FINAL DATABASE
# ============================================================

consolidation_cases = load_json_list(
    FINAL_DATABASE_PATH
)


assert len(
    consolidation_cases
) == 400


# ============================================================
# NORMALIZE CASE FAMILY
# ============================================================

def get_case_family(
    case,
):
    variant = str(
        case.get(
            "case_variant",
            "",
        )
    ).lower()


    if variant == "normal":
        return "normal"


    if variant == "wrong_partner":
        return "wrong_partner"


    if variant == "silent_partner":
        return "silent_partner"


    if variant.startswith(
        "lag"
    ):
        return "lag"


    raise ValueError(
        "Unknown case_variant: "
        f"{case.get('case_variant')}"
    )


# ============================================================
# GLOBAL COUNTS
# ============================================================

case_ids = [
    str(
        case[
            "case_id"
        ]
    )

    for case
    in consolidation_cases
]


assert len(
    case_ids
) == len(
    set(
        case_ids
    )
)


family_counts = Counter(
    get_case_family(
        case
    )

    for case
    in consolidation_cases
)


expected_family_counts = {
    "normal": 100,
    "wrong_partner": 100,
    "lag": 100,
    "silent_partner": 100,
}


assert dict(
    family_counts
) == expected_family_counts


binary_label_counts = Counter(
    str(
        case[
            "gold_binary_label"
        ]
    )

    for case
    in consolidation_cases
)


assert binary_label_counts[
    "NORMAL"
] == 100


assert binary_label_counts[
    "ANOMALOUS"
] == 300


lag_variant_counts = Counter(
    str(
        case.get(
            "case_variant"
        )
    )

    for case
    in consolidation_cases

    if get_case_family(
        case
    ) == "lag"
)


# ============================================================
# PARTICIPANT AND SEMANTIC AUDITS
# ============================================================

SEGMENT_NAMES = [
    "segment_0_0_to_60_seconds",
    "segment_1_60_to_120_seconds",
]


num_participant_records_checked = 0
num_semantic_slots_checked = 0


for case in consolidation_cases:

    for (
        role,
        turns_key,
    ) in [
        (
            "participant_A",
            "participant_A_filtered_turns",
        ),

        (
            "participant_B",
            "participant_B_filtered_turns",
        ),
    ]:

        participant = case[
            role
        ]


        turns = case[
            turns_key
        ]


        assert isinstance(
            participant,
            dict,
        )


        assert isinstance(
            turns,
            list,
        )


        assert (
            "speaks"
            in participant
        )


        assert isinstance(
            participant[
                "speaks"
            ],
            bool,
        )


        assert (
            participant[
                "speaks"
            ]
            ==
            (
                len(
                    turns
                ) > 0
            )
        ), (
            "Participant-level speaks disagrees with "
            f'VAD turns in {case["case_id"]} / {role}'
        )


        participant_semantics = (
            case[
                "semantic_summaries"
            ][role]
        )


        for segment_name in SEGMENT_NAMES:

            segment_record = (
                participant_semantics[
                    segment_name
                ]
            )


            coarse_summary = (
                segment_record.get(
                    "coarse_summary"
                )
            )


            focused_summary = (
                segment_record.get(
                    "focused_summary"
                )
            )


            assert isinstance(
                coarse_summary,
                dict,
            ), (
                "Missing coarse summary in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            assert isinstance(
                focused_summary,
                dict,
            ), (
                "Missing focused summary in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            assert (
                "speaks"
                not in focused_summary
            ), (
                "Legacy focused-summary speaks field found in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            num_semantic_slots_checked += 1


        num_participant_records_checked += 1


# ============================================================
# DISPLAY DATABASE COMPOSITION
# ============================================================

family_count_df = pd.DataFrame([
    {
        "case_family": family,
        "num_cases": count,
    }

    for family, count
    in sorted(
        family_counts.items()
    )
])


binary_count_df = pd.DataFrame([
    {
        "gold_binary_label": label,
        "num_cases": count,
    }

    for label, count
    in sorted(
        binary_label_counts.items()
    )
])


lag_variant_df = pd.DataFrame([
    {
        "case_variant": variant,
        "num_cases": count,
    }

    for variant, count
    in sorted(
        lag_variant_counts.items()
    )
])


print("=" * 88)
print("FINAL CONSOLIDATION DATABASE AUDIT PASSED")
print("=" * 88)

print(
    "Database:",
    FINAL_DATABASE_PATH,
)

print(
    "Total unique cases:",
    len(
        consolidation_cases
    ),
)

print(
    "Participant records checked:",
    num_participant_records_checked,
)

print(
    "Semantic participant-segment slots checked:",
    num_semantic_slots_checked,
)


print("\nCASE FAMILIES")

display(
    family_count_df
)


print("\nBINARY LABELS")

display(
    binary_count_df
)


print("\nLAG VARIANTS STORED IN THE FINAL DATABASE")

display(
    lag_variant_df
)

FINAL CONSOLIDATION DATABASE AUDIT PASSED
Database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
Total unique cases: 400
Participant records checked: 800
Semantic participant-segment slots checked: 1600

CASE FAMILIES


,case_family,num_cases
0,lag,100
1,normal,100
2,silent_partner,100
3,wrong_partner,100



BINARY LABELS


,gold_binary_label,num_cases
0,ANOMALOUS,300
1,NORMAL,100



LAG VARIANTS STORED IN THE FINAL DATABASE


,case_variant,num_cases
0,lag_2sec,50
1,lag_3sec,50


## 5. Inspect Participant-Level Participation Evidence

The participant-level `speaks` indicator is derived from the final filtered VAD evidence rather than from free-form visual judgement.

This field provides the explicit participation grounding developed in the isolated Silent Partner branch.

In [ ]:
# ============================================================
# PARTICIPANT-LEVEL SPEAKS SUMMARY
# ============================================================

speaks_rows = []


for case in consolidation_cases:

    speaks_rows.append({
        "case_family": (
            get_case_family(
                case
            )
        ),

        "case_id": (
            case[
                "case_id"
            ]
        ),

        "participant_A_speaks": (
            case[
                "participant_A"
            ][
                "speaks"
            ]
        ),

        "participant_B_speaks": (
            case[
                "participant_B"
            ][
                "speaks"
            ]
        ),

        "participant_A_num_turns": len(
            case[
                "participant_A_filtered_turns"
            ]
        ),

        "participant_B_num_turns": len(
            case[
                "participant_B_filtered_turns"
            ]
        ),
    })


speaks_df = pd.DataFrame(
    speaks_rows
)


speaks_summary_df = (
    speaks_df
    .groupby(
        "case_family",
        as_index=False,
    )
    .agg(
        total_cases=(
            "case_id",
            "size",
        ),

        participant_A_speaks_true=(
            "participant_A_speaks",
            "sum",
        ),

        participant_B_speaks_true=(
            "participant_B_speaks",
            "sum",
        ),
    )
)


speaks_summary_df[
    "participant_A_speaks_false"
] = (
    speaks_summary_df[
        "total_cases"
    ]
    -
    speaks_summary_df[
        "participant_A_speaks_true"
    ]
)


speaks_summary_df[
    "participant_B_speaks_false"
] = (
    speaks_summary_df[
        "total_cases"
    ]
    -
    speaks_summary_df[
        "participant_B_speaks_true"
    ]
)


speaks_summary_df = speaks_summary_df[
    [
        "case_family",
        "total_cases",

        "participant_A_speaks_true",
        "participant_A_speaks_false",

        "participant_B_speaks_true",
        "participant_B_speaks_false",
    ]
]


display(
    speaks_summary_df
)

,case_family,total_cases,participant_A_speaks_true,participant_A_speaks_false,participant_B_speaks_true,participant_B_speaks_false
0,lag,100,100,0,100,0
1,normal,100,100,0,100,0
2,silent_partner,100,100,0,0,100
3,wrong_partner,100,100,0,100,0


## 6. Optional Structured Case Inspection

The interactive inspector exposes the complete structured evidence packet for any selected NORMAL, WRONG PARTNER, LAG, or SILENT PARTNER case.

This is an auditing utility only. Case identifiers and gold metadata displayed by the inspector are **not** supplied to Qwen during consolidation inference.

In [ ]:
# ============================================================
# INTERACTIVE DATABASE INSPECTOR
# ============================================================

import ipywidgets as widgets

from IPython.display import (
    clear_output,
    display,
)


try:

    from google.colab import output

    output.enable_custom_widget_manager()

except Exception:

    pass


# ============================================================
# SPLIT AND SORT CASES
# ============================================================

cases_by_family = {
    "normal": [],
    "wrong_partner": [],
    "lag": [],
    "silent_partner": [],
}


for case in consolidation_cases:

    cases_by_family[
        get_case_family(
            case
        )
    ].append(
        case
    )


for family in cases_by_family:

    cases_by_family[
        family
    ] = sorted(
        cases_by_family[
            family
        ],

        key=lambda case: str(
            case.get(
                "case_id",
                "",
            )
        ),
    )


# ============================================================
# DISPLAY HELPERS
# ============================================================

def participant_row(
    case,
    role,
):
    participant = case[
        role
    ]


    turns_key = (
        f"{role}_filtered_turns"
    )


    turns = case.get(
        turns_key,
        [],
    )


    return {
        "role": role,

        "conversation_id": (
            participant.get(
                "conversation_id"
            )
        ),

        "participant_id": (
            participant.get(
                "participant_id"
            )
        ),

        "speaks": (
            participant.get(
                "speaks"
            )
        ),

        "num_filtered_turns": len(
            turns
        ),

        "metadata_path": (
            participant.get(
                "metadata_path"
            )
        ),
    }


def display_semantics(
    case,
    role,
):
    print(
        "\n" + "-" * 88
    )

    print(
        f"{role.upper()} SEMANTIC SUMMARIES"
    )

    print(
        "-" * 88
    )


    participant_semantics = (
        case[
            "semantic_summaries"
        ][role]
    )


    for segment_name in SEGMENT_NAMES:

        segment_record = (
            participant_semantics[
                segment_name
            ]
        )


        print(
            f"\nSEGMENT: {segment_name}"
        )


        print(
            "\nCoarse summary:"
        )


        print(
            json.dumps(
                segment_record[
                    "coarse_summary"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\nFocused summary:"
        )


        print(
            json.dumps(
                segment_record[
                    "focused_summary"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )


# ============================================================
# MAIN INSPECTION FUNCTION
# ============================================================

def inspect_case(
    family,
    sample_index,
    show_turns=True,
    show_temporal_features=True,
    show_semantics=True,
    show_full_json=False,
):
    assert family in cases_by_family


    family_cases = cases_by_family[
        family
    ]


    sample_index = int(
        sample_index
    )


    assert (
        0
        <= sample_index
        < len(
            family_cases
        )
    )


    case = family_cases[
        sample_index
    ]


    print("=" * 88)
    print("CONSOLIDATION CASE INSPECTION")
    print("=" * 88)

    print(
        "Family:",
        family,
    )

    print(
        "Family position:",
        f"{sample_index + 1}/{len(family_cases)}",
    )

    print(
        "case_id:",
        case.get(
            "case_id"
        ),
    )

    print(
        "source_group_id:",
        case.get(
            "source_group_id"
        ),
    )

    print(
        "case_variant:",
        case.get(
            "case_variant"
        ),
    )

    print(
        "gold_binary_label:",
        case.get(
            "gold_binary_label"
        ),
    )

    print(
        "gold_anomaly_type:",
        case.get(
            "gold_anomaly_type"
        ),
    )

    print(
        "A_source_conversation:",
        case.get(
            "A_source_conversation"
        ),
    )

    print(
        "B_source_conversation:",
        case.get(
            "B_source_conversation"
        ),
    )


    print(
        "\nPARTICIPANTS"
    )


    display(
        pd.DataFrame([
            participant_row(
                case,
                "participant_A",
            ),

            participant_row(
                case,
                "participant_B",
            ),
        ])
    )


    print(
        "\nSEMANTIC COVERAGE"
    )


    print(
        json.dumps(
            case.get(
                "semantic_summary_coverage"
            ),
            indent=2,
            ensure_ascii=False,
        )
    )


    if show_turns:

        print(
            "\n" + "=" * 88
        )

        print(
            "PARTICIPANT A FILTERED TURNS"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "participant_A_filtered_turns"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\n" + "=" * 88
        )

        print(
            "PARTICIPANT B FILTERED TURNS"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "participant_B_filtered_turns"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


    if show_temporal_features:

        print(
            "\n" + "=" * 88
        )

        print(
            "LOCAL TEMPORAL FEATURES"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "local_temporal_features"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\n" + "=" * 88
        )

        print(
            "GLOBAL SHIFT FEATURES"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "global_shift_features"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


    if show_semantics:

        display_semantics(
            case,
            "participant_A",
        )


        display_semantics(
            case,
            "participant_B",
        )


    if show_full_json:

        print(
            "\n" + "=" * 88
        )

        print(
            "COMPLETE SAMPLE JSON"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case,
                indent=2,
                ensure_ascii=False,
            )
        )


    return case


# ============================================================
# WIDGETS
# ============================================================

family_labels = {
    "normal": "NORMAL",
    "wrong_partner": "WRONG_PARTNER",
    "lag": "LAG",
    "silent_partner": "SILENT_PARTNER",
}


family_dropdown = widgets.Dropdown(
    options=[
        (
            family_labels[
                family
            ],
            family,
        )

        for family in [
            "normal",
            "wrong_partner",
            "lag",
            "silent_partner",
        ]
    ],

    value="normal",

    description="Family:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="450px"
    ),
)


case_dropdown = widgets.Dropdown(
    description="Sample:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="900px"
    ),
)


show_turns_checkbox = widgets.Checkbox(
    value=True,
    description="Show filtered turns",
    indent=False,
)


show_temporal_checkbox = widgets.Checkbox(
    value=True,
    description="Show temporal features",
    indent=False,
)


show_semantics_checkbox = widgets.Checkbox(
    value=True,
    description="Show semantic summaries",
    indent=False,
)


show_full_json_checkbox = widgets.Checkbox(
    value=False,
    description="Show complete JSON",
    indent=False,
)


inspector_output = widgets.Output()


def refresh_case_options(
    *args,
):
    family = family_dropdown.value


    family_cases = cases_by_family[
        family
    ]


    case_dropdown.options = [
        (
            (
                f"{index + 1:03d}/100 | "
                f'{case.get("case_id")} | '
                f'{case.get("case_variant")}'
            ),

            index,
        )

        for index, case
        in enumerate(
            family_cases
        )
    ]


    case_dropdown.value = 0


def render_case(
    *args,
):
    if case_dropdown.value is None:
        return


    with inspector_output:

        clear_output(
            wait=True
        )


        inspect_case(
            family=(
                family_dropdown.value
            ),

            sample_index=(
                case_dropdown.value
            ),

            show_turns=(
                show_turns_checkbox.value
            ),

            show_temporal_features=(
                show_temporal_checkbox.value
            ),

            show_semantics=(
                show_semantics_checkbox.value
            ),

            show_full_json=(
                show_full_json_checkbox.value
            ),
        )


family_dropdown.observe(
    refresh_case_options,
    names="value",
)


family_dropdown.observe(
    render_case,
    names="value",
)


case_dropdown.observe(
    render_case,
    names="value",
)


show_turns_checkbox.observe(
    render_case,
    names="value",
)


show_temporal_checkbox.observe(
    render_case,
    names="value",
)


show_semantics_checkbox.observe(
    render_case,
    names="value",
)


show_full_json_checkbox.observe(
    render_case,
    names="value",
)


refresh_case_options()


controls = widgets.VBox([
    family_dropdown,
    case_dropdown,

    widgets.HBox([
        show_turns_checkbox,
        show_temporal_checkbox,
    ]),

    widgets.HBox([
        show_semantics_checkbox,
        show_full_json_checkbox,
    ]),
])


print("=" * 88)
print("INTERACTIVE INSPECTOR READY")
print("=" * 88)

for family in [
    "normal",
    "wrong_partner",
    "lag",
    "silent_partner",
]:

    print(
        family,
        len(
            cases_by_family[
                family
            ]
        ),
    )


display(
    controls,
    inspector_output,
)


render_case()


INTERACTIVE INSPECTOR READY
normal 100
wrong_partner 100
lag 100
silent_partner 100


Output()

## 7. Load Qwen2.5-Omni Thinker

The final consolidation stage is text-only: Qwen2.5-Omni receives the cached structured evidence packet and produces the conversation-level decision.

In [ ]:
import torch
from transformers import Qwen2_5OmniThinkerForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)

processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)

print("Loaded:", MODEL_ID)

config.json:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/233k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1346 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-Omni-7B


## 8. Shared Qwen Inference Helpers

These functions provide deterministic text-only generation and robust JSON parsing for the binary consolidation experiments.

In [ ]:
def extract_json_from_text(text: str):
    text = text.strip()

    # remove markdown fences
    text = re.sub(r"^```(?:json)?", "", text.strip(), flags=re.IGNORECASE)
    text = re.sub(r"```$", "", text.strip())

    try:
        return json.loads(text)
    except Exception:
        pass

    # find first balanced JSON object
    start = text.find("{")
    if start == -1:
        return {"parse_error": True, "raw_output": text}

    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                candidate = text[start:i+1]
                try:
                    return json.loads(candidate)
                except Exception:
                    return {
                        "parse_error": True,
                        "raw_output": text,
                        "json_candidate": candidate,
                    }

    return {"parse_error": True, "raw_output": text}

def qwen_video_text(video_path: str, prompt: str, max_new_tokens: int = 350):
    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "You are Qwen, a virtual human developed by the Qwen Team, "
                        "Alibaba Group, capable of perceiving auditory and visual inputs, "
                        "as well as generating text and speech."
                    )
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "video", "video": video_path, "fps": 7.0},
                {"type": "text", "text": prompt},
            ],
        },
    ]

    text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    audios, images, videos = process_mm_info(
        messages,
        use_audio_in_video=True
    )

    inputs = processor(
        text=text,
        audio=audios,
        images=images,
        videos=videos,
        return_tensors="pt",
        padding=True,
        use_audio_in_video=True,
    )

    inputs = inputs.to(model.device)

    print("Input tokens:", inputs["input_ids"].shape[1])

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated

def qwen_text_only(prompt: str, max_new_tokens: int = 700):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        processor_kwargs={
            "padding": True,
        },
    )

    inputs = {
        k: v.to(model.device) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    input_token_count = int(
        inputs["input_ids"].shape[-1]
    )


    print(
        "Input tokens:",
        input_token_count,
    )


    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated


# Experiment 1 — Initial Binary Consolidation Prompt

The first unified prompt asks whether the supplied participant records are jointly consistent with a coherent 120-second dyadic interaction.

All three evidence families are available, but the prompt does **not** yet enforce an explicit independent assessment of:

- participation validity,
- semantic compatibility,
- temporal coordination.

The experiment therefore tests whether simply presenting all available evidence is sufficient for reliable unified anomaly detection.

In [ ]:
BINARY_CONSOLIDATION_PROMPT_TEMPLATE = """
You are evaluating whether two participant records are jointly
consistent with one coherent, naturally synchronized, spoken
120-second dyadic interaction, conversation.

You are given information extracted independently for Participant A
and Participant B from the same 120-second analysis timeline.

Your task is binary classification only:

- NORMAL
- ANOMALOUS

AVAILABLE EVIDENCE

You receive:

1. Whether each participant speaks during the 120-second interval.
2. Independently backchannel-filtered speech turns for each participant.
3. Local turn-handoff and overlap features.
4. Global temporal alignment-shift features.
5. Coarse semantic summaries for two synchronized 60-second segments.
6. Focused semantic summaries for the same two segments.
7. Frozen temporal reference statistics calculated only from
   separate NORMAL dyadic conversations.

You must use only the supplied evidence.

Do not infer anything from identifiers, filenames, paths, dataset order,
sample position, or hidden labels. None of those fields are provided.

============================================================
INTERPRETATION OF A NORMAL DYADIC INTERACTION
============================================================

A NORMAL case should be jointly compatible with a coherent,
naturally synchronized two-person spoken interaction.

Typical supporting evidence includes:

- Both participants contribute speech during the complete interval.
- Their turns show plausible conversational alternation.
- Silence, short pauses, limited overlap, interruptions, and natural
  variation are allowed.
- The local handoff timing is broadly compatible with the frozen
  NORMAL reference.
- The global alignment evidence does not strongly indicate that one
  participant's timeline requires a substantial correction.
- The participants' semantic summaries can plausibly belong to the
  same evolving conversation.
- They may discuss different aspects of the same subject.
- One participant may answer, elaborate on, contextualize, or react
  to the other participant.
- The topic may naturally change between Segment 0 and Segment 1.
- Exact word matching and identical topic labels are not required.

A case is ANOMALOUS when the complete evidence is not jointly
compatible with a coherent, naturally synchronized dyadic interaction.

Do not require every evidence source to be abnormal.
A strong and reliable inconsistency in one major dimension may be
sufficient, but a single noisy statistic must not determine the result.

============================================================
SPEAKS AND TURN INFORMATION
============================================================

The field "speaks" is derived directly from the final filtered VAD turns:

- speaks = true means the participant has at least one retained turn.
- speaks = false means the participant has no retained turns during
  the entire 120-second interval.

Use "speaks" together with the actual turn lists.

Do not infer speaking activity from semantic summaries.

Every turn is represented as:

[start_time, end_time]

in seconds on the same aligned 120-second timeline.

Short overlapping turns that satisfied the preprocessing definition
of backchannel-like candidates were removed independently for each
participant.

These removed turns were not semantically verified backchannels.
Therefore:

- Do not expect every natural backchannel to appear in the turn lists.
- Do not penalize a case merely because brief listener responses are absent.
- Evaluate the overall alternation and coordination pattern.

============================================================
LOCAL TEMPORAL FEATURES
============================================================

The signed strict A_end-to-B_start offsets are calculated as:

B_start minus A_end

Interpretation:

- Negative value:
  Participant B starts shortly before Participant A finishes.

- Value near zero:
  Participant B starts close to Participant A's turn boundary.

- Positive value:
  Participant B starts after Participant A finishes.

The supplied local features include:

- filtered overlap,
- the complete signed-offset list,
- number of valid offsets,
- mean,
- median,
- maximum,
- P75,
- P90,
- number and percentage of offsets above 1.5 seconds.

Use the complete distribution.

Do not classify from one maximum value, one long pause, one overlap,
or one isolated offset.

When only one or two signed offsets exist, treat the local temporal
evidence as limited.

When no signed offsets exist, do not invent offset evidence.
Use the raw turns, speaks fields, semantics, and global evidence.

============================================================
FROZEN NORMAL LOCAL-TIMING REFERENCE
============================================================

The following statistics were calculated only from a frozen set of
separate NORMAL conversations.

They are soft reference patterns, not hard thresholds.

Natural variation exists across normal conversations.

{normal_base_reference_text}

Compare the current case with the complete NORMAL reference pattern.
Do not require every current value to be close to the average.

============================================================
GLOBAL ALIGNMENT-SHIFT FEATURES
============================================================

The global alignment search evaluates hypothetical temporal corrections
to Participant B.

The observed Participant B turns supplied in the input are not changed.

Interpretation:

- best_B_correction_shift_seconds indicates the correction that produced
  the strongest bilateral turn-boundary alignment.

- A value near zero means little global correction was preferred.

- A negative value means Participant B would align better if moved
  earlier on the timeline.

- estimated_B_lateness_seconds is the non-negative lateness implied by
  the best correction.

- alignment_score_gain_vs_zero measures how much the best correction
  improves alignment compared with applying no correction.

- best_num_bilateral_events indicates how many bilateral events support
  the selected correction.

- best_event_coverage indicates how much of the available interaction
  supports that estimate.

A large correction based on very few bilateral events or very low
coverage is weak evidence.

Do not use the correction value alone.
Interpret it together with alignment gain, event count, event coverage,
local offsets, raw turns, and overlap.

============================================================
FROZEN NORMAL GLOBAL-ALIGNMENT REFERENCE
============================================================

The following statistics were calculated only from the same frozen
NORMAL reference set.

They are soft reference patterns, not hard thresholds.

{normal_global_reference_text}

============================================================
SEMANTIC EVIDENCE
============================================================

The 120-second interval is divided into:

- Segment 0: 0 to 60 seconds
- Segment 1: 60 to 120 seconds

For each participant and segment, you receive:

Coarse semantic information:

- speech_content_summary
- apparent_topic

Focused semantic information:

- detailed_speech_summary
- main_topic
- secondary_topics
- key_semantic_details
- summary_specificity
- unclear_content
- confidence

The semantic summaries were independently generated and may be broad,
imperfect, repetitive, or uncertain.

Evaluate semantic compatibility, not exact wording.

Supporting semantic compatibility may include:

- a shared concrete subject,
- complementary accounts of the same situation,
- a plausible question-and-response relationship,
- one participant supplying context for the other,
- compatible people, events, places, experiences, or arguments,
- a coherent topic transition across the two segments.

Broad labels such as:

- personal experiences,
- preferences,
- daily life,
- opinions,
- general discussion,
- lifestyle,
- personal well-being

are not sufficient evidence of compatibility by themselves.

The concrete details must provide a plausible shared conversational
context.

If a semantic summary is vague, uncertain, or low-confidence, treat it
as limited evidence rather than automatically supporting NORMAL or
ANOMALOUS.

Do not use semantic mismatch alone when the summaries are too generic
to support a reliable comparison.

============================================================
COMBINED DECISION
============================================================

Evaluate the complete 120-second case jointly.

Consider:

1. Whether both participants speak.
2. The structure and alternation of their filtered turns.
3. The signed local handoff distribution.
4. Filtered overlap.
5. Global alignment correction and its reliability.
6. Semantic compatibility within each synchronized segment.
7. Semantic coherence across the complete 120 seconds.
8. The frozen NORMAL temporal references.

Do not use a simple vote across features or segments.

Do not assume all NORMAL conversations follow the average exactly.

Do not classify a case as ANOMALOUS because of one isolated unusual
measurement.

Do not classify a case as NORMAL only because one evidence source
appears plausible.

Return the label that best describes the complete interaction.

============================================================
CURRENT CASE
============================================================

Analysis duration:
{duration_seconds:.2f} seconds

PARTICIPANT A

Speaks:
{participant_A_speaks}

Filtered turns:
{participant_A_turns}

PARTICIPANT B

Speaks:
{participant_B_speaks}

Filtered turns:
{participant_B_turns}

LOCAL TEMPORAL FEATURES

{local_temporal_features}

GLOBAL ALIGNMENT-SHIFT FEATURES

{global_shift_features}

SEMANTIC SUMMARIES

{semantic_summaries}

============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include reasoning.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()

In [ ]:
# ============================================================
# BUILD AND PRINT THE SELECTED NORMAL-ONLY REFERENCE TEXT
#
# Only the frozen NORMAL profile is used.
#
# Included base metrics:
#   - num_offsets
#   - offset_mean
#   - offset_median
#   - offset_max
#   - offset_p75
#   - offset_p90
#   - percent_above_1_5
#   - clean_overlap_seconds
#
# Included global metrics:
#   - best_B_correction_shift_seconds__mean
#   - best_B_correction_shift_seconds__median
#   - estimated_B_lateness_seconds__mean
#   - estimated_B_lateness_seconds__median
#   - alignment_score_gain_vs_zero__mean
#   - alignment_score_gain_vs_zero__median
#   - best_num_bilateral_events__mean
#   - best_event_coverage__mean
#
# No other frozen statistics are exposed.
# ============================================================

import json
import math
import numbers


# ============================================================
# STRICT NORMAL PROFILE AUDIT
# ============================================================

assert isinstance(
    frozen_normal_base_statistics,
    dict,
)


assert isinstance(
    frozen_normal_shift_statistics,
    dict,
)


assert (
    frozen_normal_base_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


assert (
    frozen_normal_shift_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


# ============================================================
# NUMERIC VALIDATION HELPER
# ============================================================

def require_finite_numeric(
    record,
    field_name,
):
    """
    Read one required numeric field and verify
    that it exists and contains a finite number.
    """

    assert field_name in record, (
        f"Missing required field: {field_name}"
    )


    value = record[
        field_name
    ]


    assert isinstance(
        value,
        numbers.Real,
    ) and not isinstance(
        value,
        bool,
    ), (
        f"Expected numeric value for {field_name}, "
        f"found {type(value).__name__}: {value}"
    )


    value = float(
        value
    )


    assert math.isfinite(
        value
    ), (
        f"Non-finite value for {field_name}: {value}"
    )


    return value


def normalize_negative_zero(
    value,
):
    """
    Convert -0.0 to 0.0 for cleaner prompt text.
    """

    if abs(
        value
    ) < 1e-12:

        return 0.0


    return value


# ============================================================
# SELECT ONLY THE REQUIRED NORMAL BASE STATISTICS
# ============================================================

normal_num_offsets = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "num_offsets",
    )
)


normal_offset_mean = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_mean",
    )
)


normal_offset_median = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_median",
    )
)


normal_offset_max = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_max",
    )
)


normal_offset_p75 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_p75",
    )
)


normal_offset_p90 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_p90",
    )
)


normal_percent_above_1_5 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "percent_above_1_5",
    )
)


normal_clean_overlap_seconds = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "clean_overlap_seconds",
    )
)


# ============================================================
# SELECT ONLY THE REQUIRED NORMAL GLOBAL-SHIFT STATISTICS
#
# IMPORTANT:
# These fields use a double underscore before
# "mean" and "median".
# ============================================================

normal_best_shift_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_B_correction_shift_seconds__mean",
    )
)


normal_best_shift_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_B_correction_shift_seconds__median",
    )
)


normal_lateness_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "estimated_B_lateness_seconds__mean",
    )
)


normal_lateness_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "estimated_B_lateness_seconds__median",
    )
)


normal_alignment_gain_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "alignment_score_gain_vs_zero__mean",
    )
)


normal_alignment_gain_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "alignment_score_gain_vs_zero__median",
    )
)


normal_bilateral_events_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_num_bilateral_events__mean",
    )
)


normal_event_coverage_fraction = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_event_coverage__mean",
    )
)


# ============================================================
# CLEAN DISPLAY VALUES
# ============================================================

normal_best_shift_mean = (
    normalize_negative_zero(
        normal_best_shift_mean
    )
)


normal_best_shift_median = (
    normalize_negative_zero(
        normal_best_shift_median
    )
)


normal_lateness_mean = (
    normalize_negative_zero(
        normal_lateness_mean
    )
)


normal_lateness_median = (
    normalize_negative_zero(
        normal_lateness_median
    )
)


normal_event_coverage_percent = (
    100.0
    * normal_event_coverage_fraction
)


# ============================================================
# STORE THE EXACT VALUES USED IN THE PROMPT
# ============================================================

normal_base_reference_values = {
    "num_signed_offsets": (
        normal_num_offsets
    ),

    "mean_signed_offset_seconds": (
        normal_offset_mean
    ),

    "median_signed_offset_seconds": (
        normal_offset_median
    ),

    "maximum_signed_offset_seconds": (
        normal_offset_max
    ),

    "p75_signed_offset_seconds": (
        normal_offset_p75
    ),

    "p90_signed_offset_seconds": (
        normal_offset_p90
    ),

    "percent_offsets_above_1_5_seconds": (
        normal_percent_above_1_5
    ),

    "filtered_clean_overlap_seconds": (
        normal_clean_overlap_seconds
    ),
}


normal_global_reference_values = {
    "mean_best_B_correction_shift_seconds": (
        normal_best_shift_mean
    ),

    "median_best_B_correction_shift_seconds": (
        normal_best_shift_median
    ),

    "mean_estimated_B_lateness_seconds": (
        normal_lateness_mean
    ),

    "median_estimated_B_lateness_seconds": (
        normal_lateness_median
    ),

    "mean_alignment_score_gain_vs_zero": (
        normal_alignment_gain_mean
    ),

    "median_alignment_score_gain_vs_zero": (
        normal_alignment_gain_median
    ),

    "mean_num_bilateral_alignment_events": (
        normal_bilateral_events_mean
    ),

    "mean_bilateral_event_coverage_percent": (
        normal_event_coverage_percent
    ),
}


assert len(
    normal_base_reference_values
) == 8


assert len(
    normal_global_reference_values
) == 8


# ============================================================
# BUILD NORMAL LOCAL-TEMPORAL REFERENCE TEXT
# ============================================================

normal_base_reference_text = f"""
Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around {normal_num_offsets:.2f}.
- Mean signed offset is around {normal_offset_mean:.2f} seconds.
- Median signed offset is around {normal_offset_median:.2f} seconds.
- Maximum signed offset is around {normal_offset_max:.2f} seconds.
- P75 signed offset is around {normal_offset_p75:.2f} seconds.
- P90 signed offset is around {normal_offset_p90:.2f} seconds.
- Approximately {normal_percent_above_1_5:.1f}% of offsets are above 1.5 seconds.
- Filtered clean overlap is around {normal_clean_overlap_seconds:.2f} seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.
""".strip()


# ============================================================
# BUILD NORMAL GLOBAL-ALIGNMENT REFERENCE TEXT
# ============================================================

normal_global_reference_text = f"""
Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around {normal_best_shift_mean:.2f} seconds.
- Median best B correction shift is around {normal_best_shift_median:.2f} seconds.
- Mean estimated B lateness is around {normal_lateness_mean:.2f} seconds.
- Median estimated B lateness is around {normal_lateness_median:.2f} seconds.
- Mean alignment score gain versus zero shift is around {normal_alignment_gain_mean:.3f}.
- Median alignment score gain versus zero shift is around {normal_alignment_gain_median:.3f}.
- Mean number of bilateral alignment events is around {normal_bilateral_events_mean:.2f}.
- Mean bilateral event coverage is around {normal_event_coverage_percent:.2f}%.

These values were calculated only from the frozen NORMAL reference conversations.
They describe typical global-alignment behavior in the frozen NORMAL set and are not hard classification thresholds.
""".strip()


# ============================================================
# PRINT BOTH PROMPT SECTIONS
# ============================================================

print("=" * 88)
print("normal_base_reference_text")
print("=" * 88)

print(
    normal_base_reference_text
)


print("\n" + "=" * 88)
print("normal_global_reference_text")
print("=" * 88)

print(
    normal_global_reference_text
)


# ============================================================
# PRINT THE EXACT STRUCTURED VALUES USED
# ============================================================

print("\n" + "=" * 88)
print("EXACT SELECTED NORMAL BASE VALUES")
print("=" * 88)

print(
    json.dumps(
        normal_base_reference_values,
        indent=2,
        ensure_ascii=False,
    )
)


print("\n" + "=" * 88)
print("EXACT SELECTED NORMAL GLOBAL VALUES")
print("=" * 88)

print(
    json.dumps(
        normal_global_reference_values,
        indent=2,
        ensure_ascii=False,
    )
)


# ============================================================
# STRICT CONTENT AUDIT
# ============================================================

combined_reference_text = (
    normal_base_reference_text
    + "\n"
    + normal_global_reference_text
)


# No anomaly-specific labels or profiles.
for forbidden_text in [
    "LAG +1",
    "LAG +2",
    "LAG +3",
    "wrong_partner",
    "silent_partner",
    "anomaly_type",
]:

    assert (
        forbidden_text.lower()
        not in combined_reference_text.lower()
    )


# Unwanted frozen reference fields must not appear.
for excluded_text in [
    "original A turns",
    "original B turns",
    "removed A backchannels",
    "removed B backchannels",
    "filtered A turns",
    "filtered B turns",
    "number of negative",
    "number of positive",
    "minimum signed offset",
    "clean overlap percentage",
]:

    assert (
        excluded_text.lower()
        not in combined_reference_text.lower()
    )


# All required selected metrics must appear.
for required_text in [
    "Number of signed offsets",
    "Mean signed offset",
    "Median signed offset",
    "Maximum signed offset",
    "P75 signed offset",
    "P90 signed offset",
    "offsets are above 1.5 seconds",
    "Filtered clean overlap",
    "Mean best B correction shift",
    "Median best B correction shift",
    "Mean estimated B lateness",
    "Median estimated B lateness",
    "Mean alignment score gain",
    "Median alignment score gain",
    "Mean number of bilateral alignment events",
    "Mean bilateral event coverage",
]:

    assert (
        required_text
        in combined_reference_text
    )


print("\n" + "=" * 88)
print("SELECTED NORMAL-ONLY REFERENCE TEXT READY")
print("=" * 88)

print(
    "Base reference metrics included:",
    len(
        normal_base_reference_values
    ),
)

print(
    "Global reference metrics included:",
    len(
        normal_global_reference_values
    ),
)

print(
    "No additional frozen statistics were exposed."
)

print(
    "No explicit anomaly profile or anomaly type was included."
)

normal_base_reference_text
Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around 5.76.
- Mean signed offset is around 0.30 seconds.
- Median signed offset is around 0.26 seconds.
- Maximum signed offset is around 1.05 seconds.
- P75 signed offset is around 0.59 seconds.
- P90 signed offset is around 0.83 seconds.
- Approximately 6.7% of offsets are above 1.5 seconds.
- Filtered clean overlap is around 6.25 seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.

normal_global_reference_text
Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around -0.23 seconds.
- Median best B correction shift is around 0.00 seconds.
- Mean estimated B lateness is around 0.50 seconds.
- Median estimated B lateness is around 0.00 seconds.
- Mean alignment score gain versus zero shift is around 0.023.

## 9. Run Experiment 1 on All 400 Cases

The model receives only the selected structured evidence and the frozen NORMAL reference.

Gold labels and case metadata are attached to the saved cache **after generation** for evaluation and are never included in the model-facing prompt.

In [ ]:
# ============================================================
# BINARY-ONLY CONSOLIDATION EXPERIMENT
# INPUT PROJECTION, PROMPT BUILDING AND INFERENCE HELPERS
#
# IMPORTANT:
# The model receives only:
#   - participant-level speaks
#   - filtered turns
#   - selected local temporal features
#   - selected global alignment-shift features
#   - coarse and focused semantic summaries
#   - frozen NORMAL references
#
# It does NOT receive:
#   - case_id
#   - source_group_id
#   - participant/conversation IDs
#   - case_variant
#   - pairing_type
#   - gold labels
#   - file paths
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import copy
import hashlib
import json
import re
import time

import pandas as pd
import torch

from tqdm.auto import tqdm


# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================

EXPERIMENT_VERSION = (
    "binary_only_consolidation_normal_reference_v1"
)


EXPERIMENT_DIR = (
    OUT_DIR
    / EXPERIMENT_VERSION
)


EXPERIMENT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


PREDICTION_CACHE_PATH = (
    EXPERIMENT_DIR
    / "predictions_cache.json"
)


PREDICTIONS_CSV_PATH = (
    EXPERIMENT_DIR
    / "predictions_all_400.csv"
)


METRICS_JSON_PATH = (
    EXPERIMENT_DIR
    / "metrics.json"
)


CONFUSION_MATRIX_CSV_PATH = (
    EXPERIMENT_DIR
    / "confusion_matrix.csv"
)


FAMILY_METRICS_CSV_PATH = (
    EXPERIMENT_DIR
    / "metrics_by_case_family.csv"
)


VARIANT_METRICS_CSV_PATH = (
    EXPERIMENT_DIR
    / "metrics_by_case_variant.csv"
)


ERRORS_CSV_PATH = (
    EXPERIMENT_DIR
    / "classification_errors.csv"
)


PROMPT_TEMPLATE_PATH = (
    EXPERIMENT_DIR
    / "prompt_template.txt"
)


NORMAL_REFERENCE_PATH = (
    EXPERIMENT_DIR
    / "normal_reference_text.txt"
)


MAX_NEW_TOKENS = 64


LABELS = [
    "NORMAL",
    "ANOMALOUS",
]


# ============================================================
# EXACT SEMANTIC FIELDS PASSED TO THE MODEL
# ============================================================

COARSE_FIELDS = [
    "speech_content_summary",
    "apparent_topic",
]


FOCUSED_FIELDS = [
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


SEGMENT_MAP = {
    "segment_0_0_to_60_seconds": (
        "segment_0"
    ),

    "segment_1_60_to_120_seconds": (
        "segment_1"
    ),
}


# ============================================================
# EXACT TEMPORAL FIELDS PASSED TO THE MODEL
# ============================================================

LOCAL_FEATURE_FIELDS = [
    "clean_overlap_seconds",
    "clean_overlap_percent",

    "signed_strict_offsets_seconds",
    "num_signed_strict_offsets",

    "offset_mean_seconds",
    "offset_median_seconds",
    "offset_max_seconds",
    "offset_p75_seconds",
    "offset_p90_seconds",

    "num_offsets_above_1_5_seconds",
    "percent_offsets_above_1_5_seconds",
]


GLOBAL_FEATURE_FIELDS = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_event_coverage",
]


# ============================================================
# FIELDS THAT MUST NEVER APPEAR IN THE MODEL PROMPT
# ============================================================

FORBIDDEN_PROMPT_KEYS = [
    "case_id",
    "source_group_id",
    "pair_index",

    "gold_binary_label",
    "gold_anomaly_type",

    "case_variant",
    "pairing_type",

    "conversation_id",
    "participant_id",
    "metadata_path",

    "num_raw_vad_entries",

    "A_source_conversation",
    "B_source_conversation",

    "semantic_summary_source",
    "semantic_summary_coverage",
]


# ============================================================
# GENERAL HELPERS
# ============================================================

def canonical_json(
    value,
):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )


def sha256_text(
    text,
):
    return hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


def select_exact_fields(
    record,
    fields,
    context,
):
    assert isinstance(
        record,
        dict,
    ), (
        f"Expected dictionary for {context}"
    )


    missing_fields = [
        field

        for field in fields

        if field not in record
    ]


    assert not missing_fields, (
        f"Missing fields in {context}: "
        f"{missing_fields}"
    )


    return {
        field: copy.deepcopy(
            record[
                field
            ]
        )

        for field in fields
    }


# ============================================================
# TURN PROJECTION
#
# Database format:
#   {"start": 1.2, "end": 3.4}
#
# Prompt format:
#   [1.2, 3.4]
# ============================================================

def compact_turns(
    turns,
    context,
):
    assert isinstance(
        turns,
        list,
    ), (
        f"Expected list for {context}"
    )


    compact = []


    for index, turn in enumerate(
        turns
    ):

        assert isinstance(
            turn,
            dict,
        ), (
            f"Invalid turn in {context} "
            f"at index {index}"
        )


        assert "start" in turn
        assert "end" in turn


        start = float(
            turn[
                "start"
            ]
        )


        end = float(
            turn[
                "end"
            ]
        )


        assert end >= start


        compact.append([
            start,
            end,
        ])


    return compact


# ============================================================
# SEMANTIC INPUT PROJECTION
#
# Participant IDs and other metadata are deliberately excluded.
# ============================================================

def build_semantic_input(
    case,
):
    semantic_input = {}


    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant_semantics = (
            case[
                "semantic_summaries"
            ][role]
        )


        role_output = {}


        for (
            database_segment_name,
            prompt_segment_name,
        ) in SEGMENT_MAP.items():

            segment_record = (
                participant_semantics[
                    database_segment_name
                ]
            )


            coarse_summary = (
                select_exact_fields(
                    segment_record[
                        "coarse_summary"
                    ],

                    COARSE_FIELDS,

                    (
                        f"{role}/"
                        f"{database_segment_name}/"
                        "coarse_summary"
                    ),
                )
            )


            focused_summary = (
                select_exact_fields(
                    segment_record[
                        "focused_summary"
                    ],

                    FOCUSED_FIELDS,

                    (
                        f"{role}/"
                        f"{database_segment_name}/"
                        "focused_summary"
                    ),
                )
            )


            # Legacy focused-summary speaks must not exist.
            assert (
                "speaks"
                not in focused_summary
            )


            role_output[
                prompt_segment_name
            ] = {
                "coarse_summary": (
                    coarse_summary
                ),

                "focused_summary": (
                    focused_summary
                ),
            }


        semantic_input[
            role
        ] = role_output


    return semantic_input


# ============================================================
# BUILD THE EXACT MODEL INPUT
# ============================================================

def build_binary_model_input(
    case,
):
    local_features = (
        select_exact_fields(
            case[
                "local_temporal_features"
            ],

            LOCAL_FEATURE_FIELDS,

            "local_temporal_features",
        )
    )


    global_features_raw = (
        select_exact_fields(
            case[
                "global_shift_features"
            ],

            GLOBAL_FEATURE_FIELDS,

            "global_shift_features",
        )
    )


    # Stored in the database as a fraction.
    # Presented in the prompt as a percentage so that it is
    # directly comparable with the frozen NORMAL percentage.
    event_coverage_fraction = float(
        global_features_raw.pop(
            "best_event_coverage"
        )
    )


    global_features = {
        **global_features_raw,

        "best_event_coverage_percent": round(
            100.0
            * event_coverage_fraction,

            6,
        ),
    }


    payload = {
        "analysis_duration_seconds": float(
            case[
                "analysis_duration_seconds"
            ]
        ),

        "participant_A": {
            "speaks": bool(
                case[
                    "participant_A"
                ][
                    "speaks"
                ]
            ),

            "filtered_turns": compact_turns(
                case[
                    "participant_A_filtered_turns"
                ],

                "participant_A_filtered_turns",
            ),
        },

        "participant_B": {
            "speaks": bool(
                case[
                    "participant_B"
                ][
                    "speaks"
                ]
            ),

            "filtered_turns": compact_turns(
                case[
                    "participant_B_filtered_turns"
                ],

                "participant_B_filtered_turns",
            ),
        },

        "local_temporal_features": (
            local_features
        ),

        "global_shift_features": (
            global_features
        ),

        "semantic_summaries": (
            build_semantic_input(
                case
            )
        ),
    }


    assert set(
        payload
    ) == {
        "analysis_duration_seconds",
        "participant_A",
        "participant_B",
        "local_temporal_features",
        "global_shift_features",
        "semantic_summaries",
    }


    return payload


# ============================================================
# BUILD THE FINAL PROMPT FOR ONE CASE
# ============================================================

def build_binary_prompt(
    case,
):
    payload = build_binary_model_input(
        case
    )


    prompt = (
        BINARY_CONSOLIDATION_PROMPT_TEMPLATE
        .format(
            normal_base_reference_text=(
                normal_base_reference_text
            ),

            normal_global_reference_text=(
                normal_global_reference_text
            ),

            duration_seconds=(
                payload[
                    "analysis_duration_seconds"
                ]
            ),

            participant_A_speaks=(
                canonical_json(
                    payload[
                        "participant_A"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_A_turns=(
                canonical_json(
                    payload[
                        "participant_A"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            participant_B_speaks=(
                canonical_json(
                    payload[
                        "participant_B"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_B_turns=(
                canonical_json(
                    payload[
                        "participant_B"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            local_temporal_features=(
                json.dumps(
                    payload[
                        "local_temporal_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            global_shift_features=(
                json.dumps(
                    payload[
                        "global_shift_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            semantic_summaries=(
                json.dumps(
                    payload[
                        "semantic_summaries"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),
        )
    )


    # Strict leakage audit.
    prompt_lower = prompt.lower()


    for forbidden_key in (
        FORBIDDEN_PROMPT_KEYS
    ):

        assert (
            forbidden_key.lower()
            not in prompt_lower
        ), (
            "Forbidden field name leaked into prompt: "
            f"{forbidden_key}"
        )


    return (
        prompt,
        payload,
    )


# ============================================================
# TEXT-ONLY QWEN INFERENCE
#
# The existing qwen_text_only helper used 700 output tokens.
# Here only 64 are needed because the expected output is:
#
#   {"label": "NORMAL"}
#
# or:
#
#   {"label": "ANOMALOUS"}
# ============================================================

def qwen_text_only_binary(
    prompt,
    max_new_tokens=MAX_NEW_TOKENS,
):
    messages = [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ]


    inputs = (
        processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",

            processor_kwargs={
                "padding": True,
            },
        )
    )


    inputs = {
        key: (
            value.to(
                model.device
            )
            if hasattr(
                value,
                "to",
            )
            else value
        )

        for key, value
        in inputs.items()
    }


    input_token_count = int(
        inputs[
            "input_ids"
        ].shape[-1]
    )


    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,

            max_new_tokens=(
                max_new_tokens
            ),

            do_sample=False,

            use_cache=True,
        )


    generated_ids = output_ids[
        :,
        inputs[
            "input_ids"
        ].shape[1]:,
    ]


    raw_output = (
        processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]
    )


    return (
        raw_output,
        input_token_count,
    )


# ============================================================
# ROBUST JSON PARSING
# ============================================================

def extract_first_json_object(
    text,
):
    cleaned = text.strip()


    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )


    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned,
    )


    try:

        return json.loads(
            cleaned
        )

    except Exception:

        pass


    decoder = json.JSONDecoder()


    possible_starts = [
        index

        for index, character
        in enumerate(
            cleaned
        )

        if character == "{"
    ]


    for start in possible_starts:

        try:

            parsed, _ = (
                decoder.raw_decode(
                    cleaned[
                        start:
                    ]
                )
            )


            return parsed

        except Exception:

            continue


    return None


def parse_binary_prediction(
    raw_output,
):
    parsed = extract_first_json_object(
        raw_output
    )


    if isinstance(
        parsed,
        dict,
    ):

        label = str(
            parsed.get(
                "label",
                "",
            )
        ).strip().upper()


        if label in LABELS:

            return {
                "prediction": label,

                "parse_mode": "json",

                "schema_exact": (
                    set(
                        parsed.keys()
                    )
                    == {
                        "label"
                    }
                ),

                "parsed_output": (
                    parsed
                ),
            }


    # Very conservative fallback:
    # accept only a raw output that is exactly one label.
    plain_output = (
        raw_output
        .strip()
        .strip('"')
        .strip("'")
        .upper()
    )


    if plain_output in LABELS:

        return {
            "prediction": (
                plain_output
            ),

            "parse_mode": (
                "exact_plaintext_fallback"
            ),

            "schema_exact": False,

            "parsed_output": None,
        }


    return {
        "prediction": None,

        "parse_mode": "invalid",

        "schema_exact": False,

        "parsed_output": parsed,
    }


# ============================================================
# EXPERIMENT SIGNATURES
# ============================================================

PROMPT_TEMPLATE_SHA256 = sha256_text(
    BINARY_CONSOLIDATION_PROMPT_TEMPLATE
)


NORMAL_REFERENCE_SHA256 = sha256_text(
    normal_base_reference_text
    + "\n"
    + normal_global_reference_text
)


# Save exact prompt/reference definitions.
PROMPT_TEMPLATE_PATH.write_text(
    BINARY_CONSOLIDATION_PROMPT_TEMPLATE,
    encoding="utf-8",
)


NORMAL_REFERENCE_PATH.write_text(
    (
        normal_base_reference_text
        + "\n\n"
        + normal_global_reference_text
    ),
    encoding="utf-8",
)


# ============================================================
# DRY INPUT AUDIT — NO MODEL CALL
# ============================================================

example_prompt, example_payload = (
    build_binary_prompt(
        consolidation_cases[0]
    )
)


print("=" * 88)
print("BINARY EXPERIMENT INPUT AUDIT PASSED")
print("=" * 88)

print(
    "Experiment version:",
    EXPERIMENT_VERSION,
)

print(
    "Cases available:",
    len(
        consolidation_cases
    ),
)

print(
    "Prompt characters for first case:",
    len(
        example_prompt
    ),
)

print(
    "Prompt template SHA256:",
    PROMPT_TEMPLATE_SHA256,
)

print(
    "NORMAL reference SHA256:",
    NORMAL_REFERENCE_SHA256,
)


print(
    "\nSelected input structure "
    "(truncated display only):"
)


print(
    json.dumps(
        example_payload,
        indent=2,
        ensure_ascii=False,
    )[:6000]
)


print(
    "\nNo identifiers, paths, variants, "
    "or gold labels are included in the prompt."
)

BINARY EXPERIMENT INPUT AUDIT PASSED
Experiment version: binary_only_consolidation_normal_reference_v1
Cases available: 400
Prompt characters for first case: 16830
Prompt template SHA256: be8b12768ad45c6622d62ebf2c7f3a1a6c0a138eeb19556869fd99f374e22722
NORMAL reference SHA256: 8ada07c4141d0c3db070cbbbd9081aa7d0113001a7196cfcb8168f5b6f40e23a

Selected input structure (truncated display only):
{
  "analysis_duration_seconds": 120.0,
  "participant_A": {
    "speaks": true,
    "filtered_turns": [
      [
        9.89,
        10.78
      ],
      [
        11.65,
        18.24
      ],
      [
        25.38,
        27.71
      ],
      [
        32.83,
        33.76
      ],
      [
        35.65,
        44.83
      ],
      [
        68.67,
        75.01
      ],
      [
        75.81,
        76.67
      ],
      [
        78.75,
        84.67
      ],
      [
        85.57,
        90.49
      ]
    ]
  },
  "participant_B": {
    "speaks": true,
    "filtered_turns": [
      [
    

In [ ]:
# ============================================================
# RUN BINARY-ONLY INFERENCE ON ALL 400 CASES
#
# Resume behavior:
# - completed valid predictions are reused
# - generation errors or invalid outputs are retried
#
# Gold labels and case metadata are written to the cache
# only AFTER inference, exclusively for evaluation.
# They are never included in the model prompt.
# ============================================================

def utc_now_iso():
    return (
        datetime.now(
            timezone.utc
        ).isoformat()
    )


def atomic_write_json(
    path,
    data,
):
    path = Path(
        path
    )


    temporary_path = (
        path.with_suffix(
            path.suffix
            + ".tmp"
        )
    )


    temporary_path.write_text(
        json.dumps(
            data,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    temporary_path.replace(
        path
    )


def new_prediction_cache():
    return {
        "experiment_version": (
            EXPERIMENT_VERSION
        ),

        "model_id": (
            MODEL_ID
        ),

        "prompt_template_sha256": (
            PROMPT_TEMPLATE_SHA256
        ),

        "normal_reference_sha256": (
            NORMAL_REFERENCE_SHA256
        ),

        "created_at_utc": (
            utc_now_iso()
        ),

        "updated_at_utc": (
            utc_now_iso()
        ),

        "records": {},
    }


# ============================================================
# LOAD OR CREATE CACHE
# ============================================================

if PREDICTION_CACHE_PATH.exists():

    prediction_cache = json.loads(
        PREDICTION_CACHE_PATH.read_text(
            encoding="utf-8"
        )
    )


    assert (
        prediction_cache[
            "experiment_version"
        ]
        == EXPERIMENT_VERSION
    )


    assert (
        prediction_cache[
            "model_id"
        ]
        == MODEL_ID
    )


    assert (
        prediction_cache[
            "prompt_template_sha256"
        ]
        == PROMPT_TEMPLATE_SHA256
    )


    assert (
        prediction_cache[
            "normal_reference_sha256"
        ]
        == NORMAL_REFERENCE_SHA256
    )


    assert isinstance(
        prediction_cache[
            "records"
        ],
        dict,
    )


    print(
        "Resuming cache:",
        PREDICTION_CACHE_PATH,
    )


    print(
        "Existing records:",
        len(
            prediction_cache[
                "records"
            ]
        ),
    )


else:

    prediction_cache = (
        new_prediction_cache()
    )


    atomic_write_json(
        PREDICTION_CACHE_PATH,
        prediction_cache,
    )


    print(
        "Created cache:",
        PREDICTION_CACHE_PATH,
    )


# ============================================================
# DETERMINISTIC CASE ORDER
# ============================================================

ordered_cases = sorted(
    consolidation_cases,

    key=lambda case: str(
        case[
            "case_id"
        ]
    ),
)


assert len(
    ordered_cases
) == 400


# ============================================================
# RUN
# ============================================================

for case in tqdm(
    ordered_cases,
    desc="Binary-only consolidation",
):

    case_id = str(
        case[
            "case_id"
        ]
    )


    # Build the input without labels or identifiers.
    prompt, input_payload = (
        build_binary_prompt(
            case
        )
    )


    prompt_sha256 = sha256_text(
        prompt
    )


    input_payload_sha256 = (
        sha256_text(
            canonical_json(
                input_payload
            )
        )
    )


    existing_record = (
        prediction_cache[
            "records"
        ].get(
            case_id
        )
    )


    # Reuse only an already valid prediction generated
    # from exactly the same prompt and input.
    if (
        existing_record is not None
        and existing_record.get(
            "prediction"
        ) in LABELS
    ):

        assert (
            existing_record[
                "prompt_sha256"
            ]
            == prompt_sha256
        )


        assert (
            existing_record[
                "input_payload_sha256"
            ]
            == input_payload_sha256
        )


        continue


    started = time.perf_counter()


    try:

        raw_output, input_token_count = (
            qwen_text_only_binary(
                prompt,
                max_new_tokens=(
                    MAX_NEW_TOKENS
                ),
            )
        )


        parsed_result = (
            parse_binary_prediction(
                raw_output
            )
        )


        generation_error = None


    except Exception as exc:

        raw_output = ""


        input_token_count = None


        parsed_result = {
            "prediction": None,

            "parse_mode": (
                "generation_error"
            ),

            "schema_exact": False,

            "parsed_output": None,
        }


        generation_error = (
            f"{type(exc).__name__}: "
            f"{exc}"
        )


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


    elapsed_seconds = (
        time.perf_counter()
        - started
    )


    # IMPORTANT:
    # Gold labels and family names are attached only now,
    # after model generation, for evaluation and bookkeeping.
    prediction_cache[
        "records"
    ][case_id] = {
        "case_id": (
            case_id
        ),

        "source_group_id": str(
            case[
                "source_group_id"
            ]
        ),

        "case_family": (
            get_case_family(
                case
            )
        ),

        "case_variant": str(
            case[
                "case_variant"
            ]
        ),

        "gold_binary_label": str(
            case[
                "gold_binary_label"
            ]
        ).upper(),

        "prompt_sha256": (
            prompt_sha256
        ),

        "input_payload_sha256": (
            input_payload_sha256
        ),

        "input_token_count": (
            input_token_count
        ),

        "max_new_tokens": (
            MAX_NEW_TOKENS
        ),

        "raw_output": (
            raw_output
        ),

        "prediction": (
            parsed_result[
                "prediction"
            ]
        ),

        "parse_mode": (
            parsed_result[
                "parse_mode"
            ]
        ),

        "schema_exact": bool(
            parsed_result[
                "schema_exact"
            ]
        ),

        "parsed_output": (
            parsed_result[
                "parsed_output"
            ]
        ),

        "generation_error": (
            generation_error
        ),

        "elapsed_seconds": round(
            elapsed_seconds,
            4,
        ),

        "completed_at_utc": (
            utc_now_iso()
        ),
    }


    prediction_cache[
        "updated_at_utc"
    ] = utc_now_iso()


    # Checkpoint after every sample.
    atomic_write_json(
        PREDICTION_CACHE_PATH,
        prediction_cache,
    )


print("\n" + "=" * 88)
print("BINARY-ONLY INFERENCE COMPLETE")
print("=" * 88)

print(
    "Cached records:",
    len(
        prediction_cache[
            "records"
        ]
    ),
)

print(
    "Prediction cache:",
    PREDICTION_CACHE_PATH,
)

Resuming cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/binary_only_consolidation_normal_reference_v1/predictions_cache.json
Existing records: 400


Binary-only consolidation:   0%|          | 0/400 [00:00<?, ?it/s]


BINARY-ONLY INFERENCE COMPLETE
Cached records: 400
Prediction cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/binary_only_consolidation_normal_reference_v1/predictions_cache.json


In [ ]:
# ============================================================
# EVALUATE BINARY-ONLY CONSOLIDATION PREDICTIONS
#
# Label order:
#
# Rows:
#   Gold NORMAL
#   Gold ANOMALOUS
#
# Columns:
#   Pred NORMAL
#   Pred ANOMALOUS
#
# ANOMALOUS is treated as the positive class.
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)


# ============================================================
# RELOAD CACHE
# ============================================================

prediction_cache = json.loads(
    PREDICTION_CACHE_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    prediction_cache[
        "experiment_version"
    ]
    == EXPERIMENT_VERSION
)


assert (
    prediction_cache[
        "prompt_template_sha256"
    ]
    == PROMPT_TEMPLATE_SHA256
)


assert (
    prediction_cache[
        "normal_reference_sha256"
    ]
    == NORMAL_REFERENCE_SHA256
)


# ============================================================
# BUILD EVALUATION TABLE
# ============================================================

result_rows = []


for case in consolidation_cases:

    case_id = str(
        case[
            "case_id"
        ]
    )


    record = (
        prediction_cache[
            "records"
        ].get(
            case_id
        )
    )


    result_rows.append({
        "case_id": (
            case_id
        ),

        "source_group_id": str(
            case[
                "source_group_id"
            ]
        ),

        "case_family": (
            get_case_family(
                case
            )
        ),

        "case_variant": str(
            case[
                "case_variant"
            ]
        ),

        "gold_label": str(
            case[
                "gold_binary_label"
            ]
        ).upper(),

        "prediction": (
            None
            if record is None
            else record.get(
                "prediction"
            )
        ),

        "parse_mode": (
            "missing"
            if record is None
            else record.get(
                "parse_mode"
            )
        ),

        "schema_exact": (
            False
            if record is None
            else bool(
                record.get(
                    "schema_exact",
                    False,
                )
            )
        ),

        "input_token_count": (
            None
            if record is None
            else record.get(
                "input_token_count"
            )
        ),

        "elapsed_seconds": (
            None
            if record is None
            else record.get(
                "elapsed_seconds"
            )
        ),

        "raw_output": (
            ""
            if record is None
            else record.get(
                "raw_output",
                "",
            )
        ),

        "generation_error": (
            None
            if record is None
            else record.get(
                "generation_error"
            )
        ),
    })


results_df = pd.DataFrame(
    result_rows
)


results_df[
    "valid_prediction"
] = (
    results_df[
        "prediction"
    ].isin(
        LABELS
    )
)


results_df[
    "correct"
] = (
    results_df[
        "valid_prediction"
    ]
    &
    (
        results_df[
            "gold_label"
        ]
        ==
        results_df[
            "prediction"
        ]
    )
)


assert len(
    results_df
) == 400


assert (
    results_df[
        "case_id"
    ].is_unique
)


assert set(
    results_df[
        "gold_label"
    ]
) == set(
    LABELS
)


valid_df = (
    results_df[
        results_df[
            "valid_prediction"
        ]
    ].copy()
)


invalid_df = (
    results_df[
        ~results_df[
            "valid_prediction"
        ]
    ].copy()
)


assert len(
    valid_df
) > 0, (
    "No valid predictions are available."
)


# ============================================================
# OVERALL CONFUSION MATRIX
# ============================================================

y_true = valid_df[
    "gold_label"
]


y_pred = valid_df[
    "prediction"
]


confusion = confusion_matrix(
    y_true,
    y_pred,
    labels=LABELS,
)


confusion_df = pd.DataFrame(
    confusion,

    index=[
        "Gold NORMAL",
        "Gold ANOMALOUS",
    ],

    columns=[
        "Pred NORMAL",
        "Pred ANOMALOUS",
    ],
)


# For labels [NORMAL, ANOMALOUS]:
#
# [[TN, FP],
#  [FN, TP]]

true_normal = int(
    confusion[
        0,
        0,
    ]
)


false_anomalous = int(
    confusion[
        0,
        1,
    ]
)


false_normal = int(
    confusion[
        1,
        0,
    ]
)


true_anomalous = int(
    confusion[
        1,
        1,
    ]
)


normal_recall = (
    true_normal
    /
    (
        true_normal
        + false_anomalous
    )
    if (
        true_normal
        + false_anomalous
    )
    else float(
        "nan"
    )
)


# ============================================================
# OVERALL METRICS
# ============================================================

metrics = {
    "experiment_version": (
        EXPERIMENT_VERSION
    ),

    "total_cases": int(
        len(
            results_df
        )
    ),

    "valid_predictions": int(
        len(
            valid_df
        )
    ),

    "invalid_predictions": int(
        len(
            invalid_df
        )
    ),

    "exact_json_schema_outputs": int(
        results_df[
            "schema_exact"
        ].sum()
    ),

    "exact_json_schema_rate": float(
        results_df[
            "schema_exact"
        ].mean()
    ),

    "accuracy_valid_predictions": float(
        accuracy_score(
            y_true,
            y_pred,
        )
    ),

    # Invalid/missing predictions count as incorrect.
    "strict_accuracy_invalid_as_wrong": float(
        results_df[
            "correct"
        ].mean()
    ),

    "balanced_accuracy": float(
        balanced_accuracy_score(
            y_true,
            y_pred,
        )
    ),

    "anomalous_precision": float(
        precision_score(
            y_true,
            y_pred,
            pos_label="ANOMALOUS",
            zero_division=0,
        )
    ),

    "anomalous_recall": float(
        recall_score(
            y_true,
            y_pred,
            pos_label="ANOMALOUS",
            zero_division=0,
        )
    ),

    "anomalous_f1": float(
        f1_score(
            y_true,
            y_pred,
            pos_label="ANOMALOUS",
            zero_division=0,
        )
    ),

    "normal_recall_specificity": float(
        normal_recall
    ),

    "matthews_correlation_coefficient": float(
        matthews_corrcoef(
            y_true,
            y_pred,
        )
    ),

    # Dataset contains 100 NORMAL and 300 ANOMALOUS.
    "majority_anomalous_accuracy_baseline": (
        0.75
    ),

    "majority_anomalous_balanced_accuracy_baseline": (
        0.50
    ),

    "confusion_matrix_label_order": (
        LABELS
    ),

    "confusion_matrix": (
        confusion.tolist()
    ),
}


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

classification_report_df = (
    pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=LABELS,
            output_dict=True,
            zero_division=0,
        )
    ).T
)


# ============================================================
# GROUP-SUMMARY HELPER
# ============================================================

def summarize_result_group(
    group,
):
    valid_group = (
        group[
            group[
                "valid_prediction"
            ]
        ]
    )


    return {
        "total_cases": int(
            len(
                group
            )
        ),

        "valid_predictions": int(
            len(
                valid_group
            )
        ),

        "invalid_predictions": int(
            len(
                group
            )
            -
            len(
                valid_group
            )
        ),

        "predicted_NORMAL": int(
            (
                valid_group[
                    "prediction"
                ]
                == "NORMAL"
            ).sum()
        ),

        "predicted_ANOMALOUS": int(
            (
                valid_group[
                    "prediction"
                ]
                == "ANOMALOUS"
            ).sum()
        ),

        "correct_predictions": int(
            group[
                "correct"
            ].sum()
        ),

        "accuracy_on_valid": (
            float(
                (
                    valid_group[
                        "gold_label"
                    ]
                    ==
                    valid_group[
                        "prediction"
                    ]
                ).mean()
            )
            if len(
                valid_group
            )
            else float(
                "nan"
            )
        ),

        "strict_accuracy_invalid_as_wrong": float(
            group[
                "correct"
            ].mean()
        ),

        "exact_schema_rate": float(
            group[
                "schema_exact"
            ].mean()
        ),
    }


# ============================================================
# RESULTS BY CASE FAMILY
# ============================================================

family_metric_rows = []


for (
    family_name,
    family_group,
) in results_df.groupby(
    "case_family",
    sort=True,
):

    family_metric_rows.append({
        "case_family": (
            family_name
        ),

        **summarize_result_group(
            family_group
        ),
    })


family_metrics_df = pd.DataFrame(
    family_metric_rows
)


# ============================================================
# RESULTS BY EXACT CASE VARIANT
#
# This separates the two LAG variants if their case_variant
# strings are different.
# ============================================================

variant_metric_rows = []


for (
    variant_name,
    variant_group,
) in results_df.groupby(
    "case_variant",
    sort=True,
):

    variant_metric_rows.append({
        "case_variant": (
            variant_name
        ),

        **summarize_result_group(
            variant_group
        ),
    })


variant_metrics_df = pd.DataFrame(
    variant_metric_rows
)


# ============================================================
# MATCHED SOURCE-GROUP EXACT ACCURACY
#
# Each source group should contain:
#   - one NORMAL
#   - one WRONG_PARTNER
#   - one LAG
#   - one SILENT_PARTNER
#
# A group is correct only when all four cases are correct.
# ============================================================

group_exact_rows = []


for (
    source_group_id,
    source_group,
) in results_df.groupby(
    "source_group_id"
):

    group_exact_rows.append({
        "source_group_id": (
            source_group_id
        ),

        "num_cases": int(
            len(
                source_group
            )
        ),

        "all_predictions_valid": bool(
            source_group[
                "valid_prediction"
            ].all()
        ),

        "all_cases_correct": bool(
            source_group[
                "valid_prediction"
            ].all()
            and
            source_group[
                "correct"
            ].all()
        ),
    })


group_exact_df = pd.DataFrame(
    group_exact_rows
)


assert len(
    group_exact_df
) == 100


assert set(
    group_exact_df[
        "num_cases"
    ]
) == {
    4
}


metrics[
    "source_groups"
] = int(
    len(
        group_exact_df
    )
)


metrics[
    "source_groups_all_predictions_valid"
] = int(
    group_exact_df[
        "all_predictions_valid"
    ].sum()
)


metrics[
    "source_groups_all_four_cases_correct"
] = int(
    group_exact_df[
        "all_cases_correct"
    ].sum()
)


metrics[
    "source_group_exact_match_rate"
] = float(
    group_exact_df[
        "all_cases_correct"
    ].mean()
)


# ============================================================
# ERRORS
# ============================================================

error_df = (
    results_df[
        (
            ~results_df[
                "valid_prediction"
            ]
        )
        |
        (
            results_df[
                "gold_label"
            ]
            !=
            results_df[
                "prediction"
            ]
        )
    ].copy()
)


# ============================================================
# SAVE RESULTS
# ============================================================

results_df.to_csv(
    PREDICTIONS_CSV_PATH,
    index=False,
)


confusion_df.to_csv(
    CONFUSION_MATRIX_CSV_PATH
)


family_metrics_df.to_csv(
    FAMILY_METRICS_CSV_PATH,
    index=False,
)


variant_metrics_df.to_csv(
    VARIANT_METRICS_CSV_PATH,
    index=False,
)


error_df.to_csv(
    ERRORS_CSV_PATH,
    index=False,
)


METRICS_JSON_PATH.write_text(
    json.dumps(
        metrics,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ============================================================
# DISPLAY FINAL RESULTS
# ============================================================

print("=" * 88)
print("BINARY-ONLY CONSOLIDATION RESULTS")
print("=" * 88)

print(
    "Total cases:",
    metrics[
        "total_cases"
    ],
)

print(
    "Valid predictions:",
    metrics[
        "valid_predictions"
    ],
)

print(
    "Invalid predictions:",
    metrics[
        "invalid_predictions"
    ],
)


print(
    "Exact JSON schema rate:",
    f'{metrics["exact_json_schema_rate"]:.4f}',
)


print(
    "Accuracy on valid predictions:",
    f'{metrics["accuracy_valid_predictions"]:.4f}',
)


print(
    "Strict accuracy, invalid counted as wrong:",
    f'{metrics["strict_accuracy_invalid_as_wrong"]:.4f}',
)


print(
    "Balanced accuracy:",
    f'{metrics["balanced_accuracy"]:.4f}',
)


print(
    "ANOMALOUS precision:",
    f'{metrics["anomalous_precision"]:.4f}',
)


print(
    "ANOMALOUS recall:",
    f'{metrics["anomalous_recall"]:.4f}',
)


print(
    "ANOMALOUS F1:",
    f'{metrics["anomalous_f1"]:.4f}',
)


print(
    "NORMAL recall / specificity:",
    f'{metrics["normal_recall_specificity"]:.4f}',
)


print(
    "Matthews correlation coefficient:",
    (
        f'{metrics["matthews_correlation_coefficient"]:.4f}'
    ),
)


print(
    "Matched source-group exact rate:",
    f'{metrics["source_group_exact_match_rate"]:.4f}',
)


print(
    "\nMajority-class baseline accuracy:",
    "0.7500",
)


print(
    "Majority-class baseline balanced accuracy:",
    "0.5000",
)


print("\nCONFUSION MATRIX")
print("Rows = gold labels")
print("Columns = predicted labels")

display(
    confusion_df
)


print("\nCLASSIFICATION REPORT")

display(
    classification_report_df
)


print("\nPER CASE FAMILY")

display(
    family_metrics_df
)


print("\nPER EXACT CASE VARIANT")

display(
    variant_metrics_df
)


print(
    "\nSaved predictions:",
    PREDICTIONS_CSV_PATH,
)

print(
    "Saved metrics:",
    METRICS_JSON_PATH,
)

print(
    "Saved confusion matrix:",
    CONFUSION_MATRIX_CSV_PATH,
)

print(
    "Saved family metrics:",
    FAMILY_METRICS_CSV_PATH,
)

print(
    "Saved variant metrics:",
    VARIANT_METRICS_CSV_PATH,
)

print(
    "Saved errors:",
    ERRORS_CSV_PATH,
)


if len(
    invalid_df
):

    print(
        "\nWARNING: The confusion matrix excludes "
        "invalid predictions."
    )


    display(
        invalid_df[
            [
                "case_id",
                "case_family",
                "case_variant",
                "parse_mode",
                "raw_output",
                "generation_error",
            ]
        ]
    )


else:

    print(
        "\nAll 400 cases produced "
        "a valid binary prediction."
    )

BINARY-ONLY CONSOLIDATION RESULTS
Total cases: 400
Valid predictions: 400
Invalid predictions: 0
Exact JSON schema rate: 1.0000
Accuracy on valid predictions: 0.6275
Strict accuracy, invalid counted as wrong: 0.6275
Balanced accuracy: 0.7483
ANOMALOUS precision: 0.9935
ANOMALOUS recall: 0.5067
ANOMALOUS F1: 0.6711
NORMAL recall / specificity: 0.9900
Matthews correlation coefficient: 0.4425
Matched source-group exact rate: 0.0300

Majority-class baseline accuracy: 0.7500
Majority-class baseline balanced accuracy: 0.5000

CONFUSION MATRIX
Rows = gold labels
Columns = predicted labels


,Pred NORMAL,Pred ANOMALOUS
Gold NORMAL,99,1
Gold ANOMALOUS,148,152



CLASSIFICATION REPORT


,precision,recall,f1-score,support
NORMAL,0.400810,0.990000,0.570605,100.0000
ANOMALOUS,0.993464,0.506667,0.671082,300.0000
accuracy,0.627500,0.627500,0.627500,0.6275
macro avg,0.697137,0.748333,0.620843,400.0000
weighted avg,0.845300,0.627500,0.645963,400.0000



PER CASE FAMILY


,case_family,total_cases,valid_predictions,invalid_predictions,predicted_NORMAL,predicted_ANOMALOUS,correct_predictions,accuracy_on_valid,strict_accuracy_invalid_as_wrong,exact_schema_rate
0,lag,100,100,0,93,7,7,0.07,0.07,1.0
1,normal,100,100,0,99,1,99,0.99,0.99,1.0
2,silent_partner,100,100,0,0,100,100,1.00,1.00,1.0
3,wrong_partner,100,100,0,55,45,45,0.45,0.45,1.0



PER EXACT CASE VARIANT


,case_variant,total_cases,valid_predictions,invalid_predictions,predicted_NORMAL,predicted_ANOMALOUS,correct_predictions,accuracy_on_valid,strict_accuracy_invalid_as_wrong,exact_schema_rate
0,lag_2sec,50,50,0,46,4,4,0.08,0.08,1.0
1,lag_3sec,50,50,0,47,3,3,0.06,0.06,1.0
2,normal,100,100,0,99,1,99,0.99,0.99,1.0
3,silent_partner,100,100,0,0,100,100,1.00,1.00,1.0
4,wrong_partner,100,100,0,55,45,45,0.45,0.45,1.0



Saved predictions: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/binary_only_consolidation_normal_reference_v1/predictions_all_400.csv
Saved metrics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/binary_only_consolidation_normal_reference_v1/metrics.json
Saved confusion matrix: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/binary_only_consolidation_normal_reference_v1/confusion_matrix.csv
Saved family metrics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/binary_only_consolidation_normal_reference_v1/metrics_by_case_family.csv
Saved variant metrics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/binary_only_consolidation_normal_reference_v1/metrics_by_case_variant.csv
Saved errors: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/binary_only_consolidation_normal_reference_v1/classification_errors.csv

All 400 cases produced a valid binary prediction.


# Experiment 2 — Binary-Only Consolidation with Independent Normality Requirements

Experiment 1 reveals a major consolidation failure: the model strongly preserves NORMAL cases but misses most LAG anomalies and many Wrong Partner cases.

The evidence itself has not changed. The isolated experiments had already shown that useful semantic and temporal signals were present. The problem is therefore how the unified reasoner is instructed to **use** those signals.

Experiment 2 changes **only the prompt-level decision policy** while preserving:

- the same 400 cases,
- the same `speaks` participation evidence,
- the same filtered turns,
- the same local temporal features,
- the same global alignment-shift features,
- the same coarse semantic summaries,
- the same focused semantic summaries,
- the same frozen NORMAL temporal reference,
- the same Qwen2.5-Omni model,
- the same deterministic generation settings,
- the same binary-only JSON output schema.

## Revised decision policy

A NORMAL conversation must independently satisfy:

1. **Participation validity**  
   Both participants must provide meaningful spoken participation.

2. **Semantic conversational compatibility**  
   The participant summaries must plausibly belong to the same interaction.

3. **Temporal coordination**  
   Local turn handoffs and global alignment must remain compatible with natural coordination relative to the frozen NORMAL reference.

These requirements are evaluated separately before the final binary decision.

Most importantly, evidence sources are not intended to cancel one another. A strong and reliable failure in one requirement is sufficient to classify the complete interaction as `ANOMALOUS`.

This small prompt-level change produces the **Binary-only consolidation result reported in the thesis**:

| Family | Correct |
|---|---:|
| NORMAL | **72/100** |
| LAG | **54/100** |
| WRONG PARTNER | **95/100** |
| SILENT PARTNER | **100/100** |
| **Overall** | **80.25%** |

This configuration becomes the binary-only member of the later output-format comparison against Structured R1, free-form rationale, and Structured R1 + rationale.

In [ ]:

# ============================================================
# EXPERIMENT 2
# BINARY CONSOLIDATION WITH INDEPENDENT NORMALITY REQUIREMENTS
#
# Same:
#   - database
#   - features
#   - model
#   - frozen NORMAL references
#   - output schema
#
# Changed:
#   - prompt decision policy only
# ============================================================

from pathlib import Path

import json


# ============================================================
# METHOD 2 PROMPT
# ============================================================

BINARY_CONSOLIDATION_PROMPT_TEMPLATE = """
You are evaluating whether two participant records are jointly
consistent with one coherent, naturally synchronized, spoken
120-second dyadic interaction.

You are given information extracted independently for Participant A
and Participant B from the same 120-second analysis timeline.

Your task is binary classification only:

- NORMAL
- ANOMALOUS

Do not predict, infer, or name a specific anomaly type.

============================================================
AVAILABLE EVIDENCE
============================================================

You receive:

1. Whether each participant speaks during the 120-second interval.
2. Independently backchannel-filtered speech turns for each participant.
3. Local turn-handoff and overlap features.
4. Global temporal alignment-shift features.
5. Coarse semantic summaries for two synchronized 60-second segments.
6. Focused semantic summaries for the same two segments.
7. Frozen temporal reference statistics calculated only from
   separate NORMAL dyadic conversations.

Use only the supplied evidence.

Do not infer anything from identifiers, filenames, paths, dataset order,
sample position, participant identity, or hidden labels.

None of those fields are provided.

============================================================
OPERATIONAL DEFINITION OF NORMAL
============================================================

A NORMAL case must be compatible with one coherent, naturally
synchronized, two-person spoken interaction.

NORMAL requires three independent properties:

1. Participation validity
2. Semantic conversational compatibility
3. Temporal coordination

Evaluate these three properties separately before making the final
binary decision.

A case should be classified as NORMAL only when all three properties
are sufficiently supported by the available evidence.

A strong and reliable failure of any one property means that the
complete interaction does not satisfy the operational definition of
NORMAL.

Do not use a simple majority vote between the three properties.

Evidence that two properties appear normal must not override a strong
and reliable failure of the third property.

In particular:

- Both participants speaking does not by itself prove NORMAL.
- Semantic compatibility does not by itself prove NORMAL.
- Temporal compatibility does not compensate for strong semantic
  incompatibility.
- Strong semantic compatibility must not cancel reliable evidence
  that the participant timelines are not normally coordinated.

============================================================
PARTICIPATION VALIDITY
============================================================

The field "speaks" is derived directly from the final filtered VAD turns:

- speaks = true means that the participant has at least one retained turn.
- speaks = false means that the participant has no retained turns during
  the entire 120-second interval.

Use the speaks fields together with the actual filtered turn lists.

Do not infer speaking activity from the semantic summaries.

For a normal spoken dyadic interaction, both participants are expected
to contribute speech during the complete interval.

Natural asymmetry is allowed:

- one participant may speak substantially more than the other,
- participants may have long listening periods,
- turn numbers and speaking durations do not need to be balanced.

However, the complete absence of retained speech from one participant
is not compatible with a normal two-person spoken interaction.

============================================================
TURN FORMAT AND BACKCHANNEL FILTERING
============================================================

Every turn is represented as:

[start_time, end_time]

in seconds on the same aligned 120-second timeline.

Short overlapping turns that satisfied the preprocessing definition
of backchannel-like candidates were removed independently for each
participant.

The removed turns were operational candidates and were not semantically
verified backchannels.

Therefore:

- Do not expect every natural listener response to remain in the lists.
- Do not penalize a case merely because brief listener responses are absent.
- Silence, pauses, interruptions, limited overlap, and natural variation
  are allowed.
- Evaluate the complete turn structure and coordination pattern.

============================================================
LOCAL TEMPORAL FEATURES
============================================================

The signed strict A_end-to-B_start offsets are calculated as:

B_start minus A_end

Interpretation:

Negative value:

- Participant B starts shortly before Participant A finishes.

Value near zero:

- Participant B starts close to Participant A's turn boundary.

Positive value:

- Participant B starts after Participant A finishes.

The supplied local temporal evidence includes:

- filtered overlap,
- the complete signed-offset list,
- number of valid offsets,
- mean,
- median,
- maximum,
- P75,
- P90,
- number of offsets above 1.5 seconds,
- percentage of offsets above 1.5 seconds.

Use the complete signed-offset distribution.

Do not decide from:

- one maximum value,
- one long pause,
- one overlap event,
- one negative value,
- or one isolated positive offset.

A NORMAL local temporal pattern is generally characterized by:

- repeated handoffs that are negative, near zero, or short positive,
- mean and median broadly compatible with the frozen NORMAL pattern,
- P75 and P90 broadly compatible with the frozen NORMAL pattern,
- offsets above 1.5 seconds being absent, uncommon, or isolated,
- no repeated and systematic pattern of substantially delayed handoffs.

Not every value must be negative or near zero.

A NORMAL interaction may contain occasional long pauses or unusual
handoffs.

However, repeated elevation across several parts of the offset
distribution is not equivalent to one isolated natural variation.

When the signed-offset list contains only one or two values, treat the
local temporal evidence as limited.

When the list is empty, the offset statistics are unavailable.
Do not invent missing evidence.

============================================================
FROZEN NORMAL LOCAL-TIMING REFERENCE
============================================================

The following statistics were calculated only from a frozen set of
separate NORMAL conversations:

{normal_base_reference_text}

These statistics operationally describe the expected local temporal
pattern for NORMAL interactions in this experiment.

They are not independent hard thresholds.

A current case does not need to equal every average, and natural
variation around the reference pattern is expected.

However, these statistics are not optional background information.

Evaluate the current signed-offset distribution as a whole against the
frozen NORMAL reference.

A substantial and consistent departure across multiple reliable local
features means that the local temporal requirement for NORMAL is not
satisfied.

A multi-feature departure may include several of the following
appearing together:

- substantially elevated mean,
- substantially elevated median,
- substantially elevated P75 or P90,
- several offsets above 1.5 seconds,
- a substantially elevated percentage above 1.5 seconds,
- repeated delayed handoffs visible in the raw turn structure.

One unusual feature alone is insufficient.

Several mutually supporting deviations are strong evidence.

============================================================
FILTERED OVERLAP
============================================================

Filtered overlap is secondary temporal evidence.

Natural overlap and interruption may occur in NORMAL interactions.

Overlap alone must not determine the label.

Interpret overlap together with:

- the signed-offset distribution,
- the raw turn structure,
- and the global alignment evidence.

============================================================
GLOBAL ALIGNMENT-SHIFT FEATURES
============================================================

The global alignment search evaluates hypothetical temporal corrections
to Participant B.

The observed Participant B turns supplied for classification are not
changed.

The global features have the following meanings:

- best_B_correction_shift_seconds is the hypothetical correction that
  produced the strongest bilateral turn-boundary alignment.

- A correction near zero means that little global temporal correction
  was preferred.

- A negative correction means that Participant B would align better if
  Participant B's timeline were moved earlier.

- estimated_B_lateness_seconds is the non-negative lateness implied by
  the best correction.

- alignment_score_gain_vs_zero measures how much the best correction
  improves alignment compared with applying no correction.

- best_num_bilateral_events indicates how many bilateral events support
  the selected correction.

- best_event_coverage indicates how much of the available interaction
  supports that estimate.

Do not use the correction value alone.

A large correction based on very few events or very low event coverage
is weak evidence.

A global estimate becomes more reliable when it is jointly supported by:

- a meaningful correction,
- meaningful estimated lateness,
- meaningful alignment improvement over zero shift,
- several bilateral events,
- sufficiently broad event coverage,
- and agreement with the local signed-offset pattern.

============================================================
FROZEN NORMAL GLOBAL-ALIGNMENT REFERENCE
============================================================

The following statistics were calculated only from the same frozen
NORMAL reference set:

{normal_global_reference_text}

These statistics operationally describe the expected global alignment
behavior for NORMAL interactions in this experiment.

They are not independent hard thresholds.

Natural variation is allowed, and one unusual global value does not
automatically exclude NORMAL.

For NORMAL temporal coordination, the global evidence is generally
expected to show:

- a preferred correction near the frozen NORMAL pattern,
- limited estimated lateness,
- limited improvement over applying no correction,
- or insufficient reliable evidence that a substantial correction
  is necessary.

The global temporal requirement for NORMAL is not satisfied when a
substantial correction is reliably supported by the evidence as a whole.

Evaluate together:

- correction direction and magnitude,
- estimated lateness,
- alignment-score gain over zero shift,
- number of bilateral events,
- event coverage,
- agreement with the local offset distribution.

A substantial correction with negligible gain, very few events, or
very low coverage is weak evidence.

A substantial correction with meaningful gain, sufficient bilateral
events, broad coverage, and matching local evidence is strong evidence
that the interaction does not follow the frozen NORMAL temporal pattern.

============================================================
JOINT TEMPORAL COORDINATION DECISION
============================================================

Local and global temporal evidence must be evaluated together.

Temporal coordination is a necessary and independent property of a
NORMAL interaction.

A case can fail the temporal requirement for NORMAL even when:

- both participants speak,
- their semantic content is compatible,
- they discuss the same topic,
- they refer to the same people or events.

Semantic compatibility establishes content compatibility.
It does not establish temporal synchronization.

Treat the temporal dimension as compatible with NORMAL when:

- the signed-offset distribution is broadly consistent with the
  frozen NORMAL local pattern,
- long positive offsets are absent, uncommon, or isolated,
- elevated local values are not repeated across the distribution,
- the preferred global correction is near the frozen NORMAL pattern,
- or a larger correction is weakly supported by gain, events, or coverage.

Treat the temporal dimension as not compatible with NORMAL when:

- multiple local distribution statistics substantially depart from
  the frozen NORMAL pattern,
- delayed handoffs form a repeated rather than isolated pattern,
- and reliable global alignment evidence supports the same conclusion.

Do not require every temporal feature to depart from NORMAL.

Do not require every signed offset to be positive or large.

Do not allow one negative or near-zero offset to cancel a broader,
well-supported non-NORMAL temporal pattern.

When reliable local and global temporal evidence agree that the
interaction substantially departs from the frozen NORMAL pattern,
the complete case must be classified as ANOMALOUS, even when the
semantic evidence is fully compatible.

Do not infer or report the cause or magnitude of the temporal failure.

============================================================
SEMANTIC EVIDENCE
============================================================

The 120-second interval is divided into:

- Segment 0: 0 to 60 seconds
- Segment 1: 60 to 120 seconds

For each participant and segment, you receive:

Coarse semantic information:

- speech_content_summary
- apparent_topic

Focused semantic information:

- detailed_speech_summary
- main_topic
- secondary_topics
- key_semantic_details
- summary_specificity
- unclear_content
- confidence

The semantic summaries were independently generated and may be broad,
imperfect, repetitive, or uncertain.

Evaluate semantic compatibility rather than exact wording.

A NORMAL semantic relationship may include:

- a shared concrete subject,
- complementary accounts of the same situation,
- a plausible question-and-response relationship,
- one participant supplying context for the other,
- one participant elaborating on or reacting to the other,
- compatible people, events, places, experiences, or arguments,
- different aspects of a shared subject,
- a coherent topic transition between Segment 0 and Segment 1.

Exact word matching and identical topic labels are not required.

A conversation may naturally change topic during the 120 seconds.

Broad labels such as:

- personal experiences,
- preferences,
- daily life,
- opinions,
- general discussion,
- lifestyle,
- personal well-being

are not sufficient evidence of semantic compatibility by themselves.

Concrete content must provide a plausible shared conversational context.

If one summary is vague, generic, unclear, or low-confidence, treat it
as limited evidence rather than automatically supporting either label.

Do not interpret the fact that both participants discuss generic
personal topics as sufficient evidence that they belong to one coherent
conversation.

Evaluate both synchronized segments and the complete 120-second semantic
relationship.

Do not use a simple vote between Segment 0 and Segment 1.

============================================================
FINAL COMBINED DECISION
============================================================

Internally evaluate:

1. Participation validity
2. Semantic conversational compatibility
3. Temporal coordination

Then make one binary decision.

Classify as NORMAL only when the complete evidence is sufficiently
compatible with all three required properties of a normal dyadic
interaction.

Classify as ANOMALOUS when at least one required property shows a strong,
reliable, and well-supported failure.

Do not require multiple properties to fail.

Do not allow strong evidence from one property to erase a reliable
failure in another property.

In particular:

- Semantic compatibility must not cancel reliable temporal failure.
- Temporal compatibility must not cancel strong semantic incompatibility.
- Speech from both participants must not cancel semantic or temporal failure.

Do not classify as ANOMALOUS because of one isolated noisy measurement.

Do not classify as NORMAL merely because one evidence source appears
plausible.

Use the reliability and consistency of the evidence, not a simple count
of supportive features.

Return one final label even when some evidence is limited.

============================================================
CURRENT CASE
============================================================

Analysis duration:
{duration_seconds:.2f} seconds

PARTICIPANT A

Speaks:
{participant_A_speaks}

Filtered turns:
{participant_A_turns}

PARTICIPANT B

Speaks:
{participant_B_speaks}

Filtered turns:
{participant_B_turns}

LOCAL TEMPORAL FEATURES

{local_temporal_features}

GLOBAL ALIGNMENT-SHIFT FEATURES

{global_shift_features}

SEMANTIC SUMMARIES

{semantic_summaries}

============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include reasoning.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


# ============================================================
# NEW EXPERIMENT VERSION AND OUTPUT DIRECTORY
# ============================================================

EXPERIMENT_VERSION = (
    "binary_only_consolidation_normal_definition_v2"
)


EXPERIMENT_DIR = (
    OUT_DIR
    / EXPERIMENT_VERSION
)


EXPERIMENT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


PREDICTION_CACHE_PATH = (
    EXPERIMENT_DIR
    / "predictions_cache.json"
)


PREDICTIONS_CSV_PATH = (
    EXPERIMENT_DIR
    / "predictions_all_400.csv"
)


METRICS_JSON_PATH = (
    EXPERIMENT_DIR
    / "metrics.json"
)


CONFUSION_MATRIX_CSV_PATH = (
    EXPERIMENT_DIR
    / "confusion_matrix.csv"
)


FAMILY_METRICS_CSV_PATH = (
    EXPERIMENT_DIR
    / "metrics_by_case_family.csv"
)


VARIANT_METRICS_CSV_PATH = (
    EXPERIMENT_DIR
    / "metrics_by_case_variant.csv"
)


ERRORS_CSV_PATH = (
    EXPERIMENT_DIR
    / "classification_errors.csv"
)


PROMPT_TEMPLATE_PATH = (
    EXPERIMENT_DIR
    / "prompt_template.txt"
)


NORMAL_REFERENCE_PATH = (
    EXPERIMENT_DIR
    / "normal_reference_text.txt"
)


MAX_NEW_TOKENS = 64


# ============================================================
# RECOMPUTE EXPERIMENT SIGNATURES
# ============================================================

PROMPT_TEMPLATE_SHA256 = sha256_text(
    BINARY_CONSOLIDATION_PROMPT_TEMPLATE
)


NORMAL_REFERENCE_SHA256 = sha256_text(
    normal_base_reference_text
    + "\n"
    + normal_global_reference_text
)


PROMPT_TEMPLATE_PATH.write_text(
    BINARY_CONSOLIDATION_PROMPT_TEMPLATE,
    encoding="utf-8",
)


NORMAL_REFERENCE_PATH.write_text(
    (
        normal_base_reference_text
        + "\n\n"
        + normal_global_reference_text
    ),
    encoding="utf-8",
)


# ============================================================
# REQUIRED STATE AUDIT
# ============================================================

required_global_names = [
    "OUT_DIR",
    "consolidation_cases",
    "model",
    "processor",

    "build_binary_prompt",
    "qwen_text_only_binary",
    "parse_binary_prediction",

    "sha256_text",
    "canonical_json",
    "get_case_family",

    "normal_base_reference_text",
    "normal_global_reference_text",
]


missing_global_names = [
    name

    for name in required_global_names

    if name not in globals()
]


assert not missing_global_names, (
    "Run the previous setup/helper cells first. "
    f"Missing variables/functions: {missing_global_names}"
)


assert len(
    consolidation_cases
) == 400


# ============================================================
# NEW PROMPT DRY AUDIT
# ============================================================

method_2_example_prompt, method_2_example_payload = (
    build_binary_prompt(
        consolidation_cases[0]
    )
)


print("=" * 88)
print("EXPERIMENT 2 CONFIGURATION READY")
print("=" * 88)

print(
    "Experiment version:",
    EXPERIMENT_VERSION,
)

print(
    "Experiment directory:",
    EXPERIMENT_DIR,
)

print(
    "Cases:",
    len(
        consolidation_cases
    ),
)

print(
    "Example prompt characters:",
    len(
        method_2_example_prompt
    ),
)

print(
    "Prompt SHA256:",
    PROMPT_TEMPLATE_SHA256,
)

print(
    "NORMAL reference SHA256:",
    NORMAL_REFERENCE_SHA256,
)

print(
    "Prediction cache:",
    PREDICTION_CACHE_PATH,
)

print(
    "Existing Method 2 cache:",
    PREDICTION_CACHE_PATH.exists(),
)

print(
    "\nThe Method 1 cache is not used by this experiment."
)

EXPERIMENT 2 CONFIGURATION READY
Experiment version: binary_only_consolidation_normal_definition_v2
Experiment directory: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/binary_only_consolidation_normal_definition_v2
Cases: 400
Example prompt characters: 23708
Prompt SHA256: ed2d68e14f1fa667650b11ef8e2c75a7f03c48c7e3faa1dfd0dabecaa14bbace
NORMAL reference SHA256: 8ada07c4141d0c3db070cbbbd9081aa7d0113001a7196cfcb8168f5b6f40e23a
Prediction cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/binary_only_consolidation_normal_definition_v2/predictions_cache.json
Existing Method 2 cache: True

The Method 1 cache is not used by this experiment.


In [ ]:
# ============================================================
# RUN EXPERIMENT 2 ON ALL 400 CASES
#
# Results are checkpointed after every case.
# Re-running this cell resumes from the Method 2 cache.
# ============================================================

from datetime import datetime, timezone

import json
import time

import torch

from tqdm.auto import tqdm


# ============================================================
# CACHE HELPERS
# ============================================================

def experiment_2_utc_now():
    return (
        datetime.now(
            timezone.utc
        ).isoformat()
    )


def experiment_2_atomic_write_json(
    path,
    data,
):
    path = Path(
        path
    )


    temporary_path = path.with_suffix(
        path.suffix
        + ".tmp"
    )


    temporary_path.write_text(
        json.dumps(
            data,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    temporary_path.replace(
        path
    )


def create_experiment_2_cache():
    return {
        "experiment_version": (
            EXPERIMENT_VERSION
        ),

        "model_id": (
            MODEL_ID
        ),

        "prompt_template_sha256": (
            PROMPT_TEMPLATE_SHA256
        ),

        "normal_reference_sha256": (
            NORMAL_REFERENCE_SHA256
        ),

        "created_at_utc": (
            experiment_2_utc_now()
        ),

        "updated_at_utc": (
            experiment_2_utc_now()
        ),

        "records": {},
    }


# ============================================================
# LOAD OR CREATE METHOD 2 CACHE
# ============================================================

if PREDICTION_CACHE_PATH.exists():

    prediction_cache = json.loads(
        PREDICTION_CACHE_PATH.read_text(
            encoding="utf-8"
        )
    )


    assert (
        prediction_cache[
            "experiment_version"
        ]
        == EXPERIMENT_VERSION
    )


    assert (
        prediction_cache[
            "model_id"
        ]
        == MODEL_ID
    )


    assert (
        prediction_cache[
            "prompt_template_sha256"
        ]
        == PROMPT_TEMPLATE_SHA256
    )


    assert (
        prediction_cache[
            "normal_reference_sha256"
        ]
        == NORMAL_REFERENCE_SHA256
    )


    print(
        "Resuming Method 2 cache:",
        PREDICTION_CACHE_PATH,
    )

    print(
        "Existing records:",
        len(
            prediction_cache[
                "records"
            ]
        ),
    )


else:

    prediction_cache = (
        create_experiment_2_cache()
    )


    experiment_2_atomic_write_json(
        PREDICTION_CACHE_PATH,
        prediction_cache,
    )


    print(
        "Created new Method 2 cache:",
        PREDICTION_CACHE_PATH,
    )


# ============================================================
# DETERMINISTIC CASE ORDER
# ============================================================

ordered_cases = sorted(
    consolidation_cases,

    key=lambda case: str(
        case[
            "case_id"
        ]
    ),
)


assert len(
    ordered_cases
) == 400


# ============================================================
# METHOD 2 INFERENCE
# ============================================================

for case in tqdm(
    ordered_cases,
    desc="Experiment 2 binary consolidation",
):

    case_id = str(
        case[
            "case_id"
        ]
    )


    prompt, model_input_payload = (
        build_binary_prompt(
            case
        )
    )


    prompt_sha256 = sha256_text(
        prompt
    )


    input_payload_sha256 = sha256_text(
        canonical_json(
            model_input_payload
        )
    )


    existing_record = (
        prediction_cache[
            "records"
        ].get(
            case_id
        )
    )


    if (
        existing_record is not None
        and existing_record.get(
            "prediction"
        ) in LABELS
    ):

        assert (
            existing_record[
                "prompt_sha256"
            ]
            == prompt_sha256
        )


        assert (
            existing_record[
                "input_payload_sha256"
            ]
            == input_payload_sha256
        )


        continue


    started = time.perf_counter()


    try:

        raw_output, input_token_count = (
            qwen_text_only_binary(
                prompt,
                max_new_tokens=(
                    MAX_NEW_TOKENS
                ),
            )
        )


        parsed_result = (
            parse_binary_prediction(
                raw_output
            )
        )


        generation_error = None


    except Exception as exc:

        raw_output = ""


        input_token_count = None


        parsed_result = {
            "prediction": None,

            "parse_mode": (
                "generation_error"
            ),

            "schema_exact": False,

            "parsed_output": None,
        }


        generation_error = (
            f"{type(exc).__name__}: {exc}"
        )


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


    elapsed_seconds = (
        time.perf_counter()
        - started
    )


    # Gold and case-family information are attached only
    # after generation for evaluation.
    prediction_cache[
        "records"
    ][case_id] = {
        "case_id": (
            case_id
        ),

        "source_group_id": str(
            case[
                "source_group_id"
            ]
        ),

        "case_family": (
            get_case_family(
                case
            )
        ),

        "case_variant": str(
            case[
                "case_variant"
            ]
        ),

        "gold_binary_label": str(
            case[
                "gold_binary_label"
            ]
        ).upper(),

        "prompt_sha256": (
            prompt_sha256
        ),

        "input_payload_sha256": (
            input_payload_sha256
        ),

        "input_token_count": (
            input_token_count
        ),

        "max_new_tokens": (
            MAX_NEW_TOKENS
        ),

        "raw_output": (
            raw_output
        ),

        "prediction": (
            parsed_result[
                "prediction"
            ]
        ),

        "parse_mode": (
            parsed_result[
                "parse_mode"
            ]
        ),

        "schema_exact": bool(
            parsed_result[
                "schema_exact"
            ]
        ),

        "parsed_output": (
            parsed_result[
                "parsed_output"
            ]
        ),

        "generation_error": (
            generation_error
        ),

        "elapsed_seconds": round(
            elapsed_seconds,
            4,
        ),

        "completed_at_utc": (
            experiment_2_utc_now()
        ),
    }


    prediction_cache[
        "updated_at_utc"
    ] = experiment_2_utc_now()


    experiment_2_atomic_write_json(
        PREDICTION_CACHE_PATH,
        prediction_cache,
    )


print("\n" + "=" * 88)
print("EXPERIMENT 2 INFERENCE COMPLETE")
print("=" * 88)

print(
    "Cached records:",
    len(
        prediction_cache[
            "records"
        ]
    ),
)

print(
    "Method 2 cache:",
    PREDICTION_CACHE_PATH,
)

Resuming Method 2 cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/binary_only_consolidation_normal_definition_v2/predictions_cache.json
Existing records: 400


Experiment 2 binary consolidation:   0%|          | 0/400 [00:00<?, ?it/s]


EXPERIMENT 2 INFERENCE COMPLETE
Cached records: 400
Method 2 cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/binary_only_consolidation_normal_definition_v2/predictions_cache.json


In [ ]:
# ============================================================
# EXPERIMENT 2 EVALUATION
# ============================================================

import json

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)

from IPython.display import display


# ============================================================
# RELOAD METHOD 2 CACHE
# ============================================================

prediction_cache = json.loads(
    PREDICTION_CACHE_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    prediction_cache[
        "experiment_version"
    ]
    == EXPERIMENT_VERSION
)


assert (
    prediction_cache[
        "prompt_template_sha256"
    ]
    == PROMPT_TEMPLATE_SHA256
)


assert (
    prediction_cache[
        "normal_reference_sha256"
    ]
    == NORMAL_REFERENCE_SHA256
)


# ============================================================
# BUILD RESULTS TABLE
# ============================================================

result_rows = []


for case in consolidation_cases:

    case_id = str(
        case[
            "case_id"
        ]
    )


    record = prediction_cache[
        "records"
    ].get(
        case_id
    )


    result_rows.append({
        "case_id": (
            case_id
        ),

        "source_group_id": str(
            case[
                "source_group_id"
            ]
        ),

        "case_family": (
            get_case_family(
                case
            )
        ),

        "case_variant": str(
            case[
                "case_variant"
            ]
        ),

        "gold_label": str(
            case[
                "gold_binary_label"
            ]
        ).upper(),

        "prediction": (
            None
            if record is None
            else record.get(
                "prediction"
            )
        ),

        "parse_mode": (
            "missing"
            if record is None
            else record.get(
                "parse_mode"
            )
        ),

        "schema_exact": (
            False
            if record is None
            else bool(
                record.get(
                    "schema_exact",
                    False,
                )
            )
        ),

        "input_token_count": (
            None
            if record is None
            else record.get(
                "input_token_count"
            )
        ),

        "elapsed_seconds": (
            None
            if record is None
            else record.get(
                "elapsed_seconds"
            )
        ),

        "raw_output": (
            ""
            if record is None
            else record.get(
                "raw_output",
                "",
            )
        ),

        "generation_error": (
            None
            if record is None
            else record.get(
                "generation_error"
            )
        ),
    })


results_df = pd.DataFrame(
    result_rows
)


results_df[
    "valid_prediction"
] = results_df[
    "prediction"
].isin(
    LABELS
)


results_df[
    "correct"
] = (
    results_df[
        "valid_prediction"
    ]
    &
    (
        results_df[
            "gold_label"
        ]
        ==
        results_df[
            "prediction"
        ]
    )
)


assert len(
    results_df
) == 400


valid_df = results_df[
    results_df[
        "valid_prediction"
    ]
].copy()


invalid_df = results_df[
    ~results_df[
        "valid_prediction"
    ]
].copy()


assert len(
    valid_df
) > 0


# ============================================================
# CONFUSION MATRIX
# ============================================================

y_true = valid_df[
    "gold_label"
]


y_pred = valid_df[
    "prediction"
]


confusion = confusion_matrix(
    y_true,
    y_pred,
    labels=[
        "NORMAL",
        "ANOMALOUS",
    ],
)


confusion_df = pd.DataFrame(
    confusion,

    index=[
        "Gold NORMAL",
        "Gold ANOMALOUS",
    ],

    columns=[
        "Pred NORMAL",
        "Pred ANOMALOUS",
    ],
)


true_normal = int(
    confusion[
        0,
        0,
    ]
)


false_anomalous = int(
    confusion[
        0,
        1,
    ]
)


normal_recall = (
    true_normal
    /
    (
        true_normal
        + false_anomalous
    )
)


# ============================================================
# OVERALL METRICS
# ============================================================

metrics = {
    "experiment_version": (
        EXPERIMENT_VERSION
    ),

    "total_cases": int(
        len(
            results_df
        )
    ),

    "valid_predictions": int(
        len(
            valid_df
        )
    ),

    "invalid_predictions": int(
        len(
            invalid_df
        )
    ),

    "exact_json_schema_rate": float(
        results_df[
            "schema_exact"
        ].mean()
    ),

    "accuracy_valid_predictions": float(
        accuracy_score(
            y_true,
            y_pred,
        )
    ),

    "strict_accuracy_invalid_as_wrong": float(
        results_df[
            "correct"
        ].mean()
    ),

    "balanced_accuracy": float(
        balanced_accuracy_score(
            y_true,
            y_pred,
        )
    ),

    "anomalous_precision": float(
        precision_score(
            y_true,
            y_pred,
            pos_label="ANOMALOUS",
            zero_division=0,
        )
    ),

    "anomalous_recall": float(
        recall_score(
            y_true,
            y_pred,
            pos_label="ANOMALOUS",
            zero_division=0,
        )
    ),

    "anomalous_f1": float(
        f1_score(
            y_true,
            y_pred,
            pos_label="ANOMALOUS",
            zero_division=0,
        )
    ),

    "normal_recall_specificity": float(
        normal_recall
    ),

    "matthews_correlation_coefficient": float(
        matthews_corrcoef(
            y_true,
            y_pred,
        )
    ),

    "confusion_matrix": (
        confusion.tolist()
    ),
}


# ============================================================
# PER-FAMILY RESULTS
# ============================================================

def summarize_method_2_group(
    group,
):
    valid_group = group[
        group[
            "valid_prediction"
        ]
    ]


    return {
        "total_cases": int(
            len(
                group
            )
        ),

        "valid_predictions": int(
            len(
                valid_group
            )
        ),

        "invalid_predictions": int(
            len(
                group
            )
            -
            len(
                valid_group
            )
        ),

        "predicted_NORMAL": int(
            (
                valid_group[
                    "prediction"
                ]
                == "NORMAL"
            ).sum()
        ),

        "predicted_ANOMALOUS": int(
            (
                valid_group[
                    "prediction"
                ]
                == "ANOMALOUS"
            ).sum()
        ),

        "correct_predictions": int(
            group[
                "correct"
            ].sum()
        ),

        "accuracy_on_valid": float(
            (
                valid_group[
                    "gold_label"
                ]
                ==
                valid_group[
                    "prediction"
                ]
            ).mean()
        ) if len(
            valid_group
        ) else float(
            "nan"
        ),

        "strict_accuracy_invalid_as_wrong": float(
            group[
                "correct"
            ].mean()
        ),

        "exact_schema_rate": float(
            group[
                "schema_exact"
            ].mean()
        ),
    }


family_rows = []


for (
    family_name,
    family_group,
) in results_df.groupby(
    "case_family",
    sort=True,
):

    family_rows.append({
        "case_family": (
            family_name
        ),

        **summarize_method_2_group(
            family_group
        ),
    })


family_metrics_df = pd.DataFrame(
    family_rows
)


variant_rows = []


for (
    variant_name,
    variant_group,
) in results_df.groupby(
    "case_variant",
    sort=True,
):

    variant_rows.append({
        "case_variant": (
            variant_name
        ),

        **summarize_method_2_group(
            variant_group
        ),
    })


variant_metrics_df = pd.DataFrame(
    variant_rows
)


classification_report_df = pd.DataFrame(
    classification_report(
        y_true,
        y_pred,
        labels=[
            "NORMAL",
            "ANOMALOUS",
        ],
        output_dict=True,
        zero_division=0,
    )
).T


error_df = results_df[
    (
        ~results_df[
            "valid_prediction"
        ]
    )
    |
    (
        results_df[
            "gold_label"
        ]
        !=
        results_df[
            "prediction"
        ]
    )
].copy()


# ============================================================
# SAVE EXPERIMENT 2 RESULTS
# ============================================================

results_df.to_csv(
    PREDICTIONS_CSV_PATH,
    index=False,
)


confusion_df.to_csv(
    CONFUSION_MATRIX_CSV_PATH
)


family_metrics_df.to_csv(
    FAMILY_METRICS_CSV_PATH,
    index=False,
)


variant_metrics_df.to_csv(
    VARIANT_METRICS_CSV_PATH,
    index=False,
)


error_df.to_csv(
    ERRORS_CSV_PATH,
    index=False,
)


METRICS_JSON_PATH.write_text(
    json.dumps(
        metrics,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ============================================================
# DISPLAY
# ============================================================

print("=" * 88)
print("EXPERIMENT 2 RESULTS")
print("=" * 88)

print(
    "Total cases:",
    metrics[
        "total_cases"
    ],
)

print(
    "Valid predictions:",
    metrics[
        "valid_predictions"
    ],
)

print(
    "Invalid predictions:",
    metrics[
        "invalid_predictions"
    ],
)

print(
    "Accuracy:",
    f'{metrics["accuracy_valid_predictions"]:.4f}',
)

print(
    "Balanced accuracy:",
    f'{metrics["balanced_accuracy"]:.4f}',
)

print(
    "ANOMALOUS precision:",
    f'{metrics["anomalous_precision"]:.4f}',
)

print(
    "ANOMALOUS recall:",
    f'{metrics["anomalous_recall"]:.4f}',
)

print(
    "ANOMALOUS F1:",
    f'{metrics["anomalous_f1"]:.4f}',
)

print(
    "NORMAL recall / specificity:",
    f'{metrics["normal_recall_specificity"]:.4f}',
)

print(
    "MCC:",
    f'{metrics["matthews_correlation_coefficient"]:.4f}',
)


print("\nCONFUSION MATRIX")

display(
    confusion_df
)


print("\nCLASSIFICATION REPORT")

display(
    classification_report_df
)


print("\nPER CASE FAMILY")

display(
    family_metrics_df
)


print("\nPER EXACT CASE VARIANT")

display(
    variant_metrics_df
)


print(
    "\nSaved Method 2 cache:",
    PREDICTION_CACHE_PATH,
)

print(
    "Saved predictions:",
    PREDICTIONS_CSV_PATH,
)

print(
    "Saved metrics:",
    METRICS_JSON_PATH,
)

print(
    "Saved errors:",
    ERRORS_CSV_PATH,
)

EXPERIMENT 2 RESULTS
Total cases: 400
Valid predictions: 400
Invalid predictions: 0
Accuracy: 0.8025
Balanced accuracy: 0.7750
ANOMALOUS precision: 0.8989
ANOMALOUS recall: 0.8300
ANOMALOUS F1: 0.8631
NORMAL recall / specificity: 0.7200
MCC: 0.5161

CONFUSION MATRIX


,Pred NORMAL,Pred ANOMALOUS
Gold NORMAL,72,28
Gold ANOMALOUS,51,249



CLASSIFICATION REPORT


,precision,recall,f1-score,support
NORMAL,0.585366,0.7200,0.645740,100.0000
ANOMALOUS,0.898917,0.8300,0.863085,300.0000
accuracy,0.802500,0.8025,0.802500,0.8025
macro avg,0.742141,0.7750,0.754412,400.0000
weighted avg,0.820529,0.8025,0.808749,400.0000



PER CASE FAMILY


,case_family,total_cases,valid_predictions,invalid_predictions,predicted_NORMAL,predicted_ANOMALOUS,correct_predictions,accuracy_on_valid,strict_accuracy_invalid_as_wrong,exact_schema_rate
0,lag,100,100,0,46,54,54,0.54,0.54,1.0
1,normal,100,100,0,72,28,72,0.72,0.72,1.0
2,silent_partner,100,100,0,0,100,100,1.00,1.00,1.0
3,wrong_partner,100,100,0,5,95,95,0.95,0.95,1.0



PER EXACT CASE VARIANT


,case_variant,total_cases,valid_predictions,invalid_predictions,predicted_NORMAL,predicted_ANOMALOUS,correct_predictions,accuracy_on_valid,strict_accuracy_invalid_as_wrong,exact_schema_rate
0,lag_2sec,50,50,0,22,28,28,0.56,0.56,1.0
1,lag_3sec,50,50,0,24,26,26,0.52,0.52,1.0
2,normal,100,100,0,72,28,72,0.72,0.72,1.0
3,silent_partner,100,100,0,0,100,100,1.00,1.00,1.0
4,wrong_partner,100,100,0,5,95,95,0.95,0.95,1.0



Saved Method 2 cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/binary_only_consolidation_normal_definition_v2/predictions_cache.json
Saved predictions: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/binary_only_consolidation_normal_definition_v2/predictions_all_400.csv
Saved metrics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/binary_only_consolidation_normal_definition_v2/metrics.json
Saved errors: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/binary_only_consolidation_normal_definition_v2/classification_errors.csv


# Binary-Only Consolidation Result

The two experiments demonstrate that **evidence availability alone is insufficient** for unified conversational anomaly detection.

The initial prompt receives the full semantic, temporal, and participation packet but achieves only **62.75% accuracy**, largely because it preserves temporal anomalies as NORMAL.

After the prompt explicitly defines participation, semantic compatibility, and temporal coordination as **independent requirements of normality**, accuracy increases to **80.25%**:

```text
NORMAL          72 / 100
LAG             54 / 100
WRONG PARTNER   95 / 100
SILENT PARTNER 100 / 100
```

No input evidence, model parameters, or output schema changed between the two experiments; only the decision instructions were revised.

This final configuration is the **Binary-only** baseline subsequently compared with the three alternative consolidation-output formats:

- Structured R1,
- free-form rationale,
- Structured R1 + rationale.